
# Predicción de Riesgo de Desnutrición Infantil
## Prueba Técnica - Especialista en Inteligencia Artificial | CEDIA




## Declaración de herramientas utilizadas

- Python
- Pandas
- Numpy
- Scikit-Learn
- XGBoost
- ChatGPT como apoyo para revisión y generación de documentación

---

## 1. Configuración inicial

- Importación de librerías
- Definición de semilla aleatoria
- Configuración de rutas
- Funciones auxiliares

---

In [1]:
# ============================================================
# PRUEBA TÉCNICA CEDIA
# Predicción de Riesgo de Desnutrición Infantil
#
# Autor: Paul Esteban Cárdenas Delgado
# Fecha: Mayo 2026
#
# Metodología:
# - CRISP-DM
# - Principios MLOps
# ============================================================

# ------------------------------------------------------------
# Librerías para manipulación de datos
# ------------------------------------------------------------

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Visualización
# ------------------------------------------------------------

import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------
# Machine Learning
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

# ------------------------------------------------------------
# Configuración general
# ------------------------------------------------------------

RANDOM_STATE = 42

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 2. Carga de datos

- Cargar X_train
- Cargar y_train
- Cargar X_test
- Cargar sample_submission
- Verificar dimensiones
- Unificar temporalmente X_train + y_train para análisis exploratorio

---


In [2]:
# ============================================================
# 1. CARGA DE DATOS
# ============================================================

# ------------------------------------------------------------
# Lectura de datasets
# ------------------------------------------------------------

X_train = pd.read_csv("X_train_Kaggle.csv")

y_train = pd.read_csv("y_train_Kaggle.csv")

X_test = pd.read_csv("X_test_Kaggle.csv")

sample_submission = pd.read_csv(
    "sample_submission_malnutrition.csv"
)



In [3]:
# ------------------------------------------------------------
# Verificación Dimensiones
# ------------------------------------------------------------

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("Sample Submission:", sample_submission.shape)

X_train: (70000, 6)
y_train: (70000, 1)
X_test: (30000, 6)
Sample Submission: (30000, 2)


In [4]:
#Información del data set (tipos de variables)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   edad_meses             70000 non-null  int64  
 1   sexo                   70000 non-null  object 
 2   region                 70000 non-null  object 
 3   nivel_educacion_madre  70000 non-null  object 
 4   peso_kg                66531 non-null  float64
 5   talla_cm               62951 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 3.2+ MB


In [5]:
# ============================================================
# CONSTRUCCIÓN DEL DATASET DE TRABAJO
# ============================================================

# ------------------------------------------------------------
# Se agrega la variable objetivo para realizar el EDA
# ------------------------------------------------------------

df = X_train.copy()

df["alerta_desnutricion"] = y_train[
    "alerta_desnutricion"
]

# ------------------------------------------------------------
# Verificación
# ------------------------------------------------------------

df.head()

,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm,alerta_desnutricion
0,0,F,Costa,Secundaria,2.243466,47.340072,0
1,11,M,Insular,Secundaria,6.157203,67.525274,0
2,22,M,Costa,Superior,9.781251,86.664662,0
3,4,F,Sierra,Primaria,2.972311,61.071857,0
4,-4,F,Costa,Primaria,NaN,50.898250,0



## 3. Revisión inicial de calidad de los datos

- Tipos de variables
- Valores únicos por variable categórica
- Valores faltantes
- Duplicados
- Rango de variables numéricas
- Detección de inconsistencias:
  - edad_meses menor a 0
  - edad_meses mayor a 24
  - peso_kg fuera de rangos promedio para Ecuador
  - talla_cm fuera de rangos promedio para Ecuador

---


###Tipos de variables

In [6]:
# ============================================================
# INFORMACIÓN GENERAL DEL DATASET
# ============================================================

print("Número de registros:", df.shape[0])

print("Número de variables:", df.shape[1])

display(df.head())


Número de registros: 70000
Número de variables: 7


,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm,alerta_desnutricion
0,0,F,Costa,Secundaria,2.243466,47.340072,0
1,11,M,Insular,Secundaria,6.157203,67.525274,0
2,22,M,Costa,Superior,9.781251,86.664662,0
3,4,F,Sierra,Primaria,2.972311,61.071857,0
4,-4,F,Costa,Primaria,NaN,50.898250,0


In [7]:
# ============================================================
# TIPOS DE VARIABLES
# ============================================================

# ------------------------------------------------------------
# Resumen de variables y tipos
# ------------------------------------------------------------

data_types = pd.DataFrame({
    "Variable": df.columns,
    "Tipo": df.dtypes.values
})

display(data_types)

,Variable,Tipo
0,edad_meses,int64
1,sexo,object
2,region,object
3,nivel_educacion_madre,object
4,peso_kg,float64
5,talla_cm,float64
6,alerta_desnutricion,int64


###Valores Faltantes

In [8]:
# ============================================================
# ANÁLISIS DE VALORES FALTANTES
# ============================================================

missing = pd.DataFrame({

    "Variable": df.columns,

    "Missing": df.isnull().sum(),

    "Porcentaje (%)":
    round(
        df.isnull().mean() * 100,
        2
    )

})

missing = missing.sort_values(
    by="Porcentaje (%)",
    ascending=False
)

display(missing)

,Variable,Missing,Porcentaje (%)
talla_cm,talla_cm,7049,10.07
peso_kg,peso_kg,3469,4.96
edad_meses,edad_meses,0,0.00
region,region,0,0.00
sexo,sexo,0,0.00
nivel_educacion_madre,nivel_educacion_madre,0,0.00
alerta_desnutricion,alerta_desnutricion,0,0.00


In [9]:
# ------------------------------------------------------------
# Visualización de valores faltantes
# ------------------------------------------------------------

fig = px.bar(

    missing,

    x="Variable",

    y="Porcentaje (%)",

    title="Porcentaje de Valores Faltantes",

    text="Porcentaje (%)"

)

fig.update_layout(

    xaxis_title="Variable",

    yaxis_title="% Missing"

)

fig.show()

In [10]:
df["peso_missing"] = (
    df["peso_kg"]
    .isna()
    .astype(int)
)

peso_missing_analysis = pd.crosstab(
    df["peso_missing"],
    df["alerta_desnutricion"],
    normalize="index"
) * 100

peso_missing_analysis

alerta_desnutricion,0,1
peso_missing,,
0,89.955059,10.044941
1,90.602479,9.397521


In [11]:
df["talla_missing"] = (
    df["talla_cm"]
    .isna()
    .astype(int)
)

talla_missing_analysis = pd.crosstab(
    df["talla_missing"],
    df["alerta_desnutricion"],
    normalize="index"
) * 100

talla_missing_analysis

alerta_desnutricion,0,1
talla_missing,,
0,89.969977,10.030023
1,90.140445,9.859555


In [12]:
df["edad_clean"] = df["edad_meses"]

df.loc[
    (df["edad_clean"] < 0) |
    (df["edad_clean"] > 24),
    "edad_clean"
] = np.nan

In [13]:
df["edad_missing"] = (
    df["edad_clean"]
    .isna()
    .astype(int)
)

edad_missing_analysis = pd.crosstab(
    df["edad_missing"],
    df["alerta_desnutricion"],
    normalize="index"
) * 100

edad_missing_analysis

alerta_desnutricion,0,1
edad_missing,,
0,89.559095,10.440905
1,93.788820,6.211180


El análisis de valores faltantes mostró que la ausencia de información en peso y talla no presenta una asociación significativa con la variable objetivo. Por tanto, no existe evidencia de que los valores faltantes aporten información predictiva adicional. En consecuencia, no se consideró necesario incorporar indicadores de ausencia (missing indicators) para estas variables.

* El dataset presenta aproximadamente 10% de valores faltantes en talla y 5% en peso.
* Se identificaron 7084 edades inconsistentes.
* Las regiones presentan una distribución equilibrada de observaciones.
* La región muestra diferencias importantes respecto al riesgo de desnutrición.
* La educación materna presenta patrones que podrían estar influenciados por el proceso sintético de generación de datos.
* Se detectaron posibles valores biológicamente improbables en peso y talla.



### Duplicados

In [14]:
duplicados = X_train.duplicated().sum()

print(
    f"Duplicados encontrados: {duplicados}"
)

Duplicados encontrados: 100


In [15]:
X_train[
    X_train.duplicated(keep=False)
].head()

,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm
469,-5,F,Sierra,Secundaria,NaN,NaN
665,9,F,Costa,Superior,NaN,NaN
769,18,F,Amazonía,Secundaria,NaN,NaN
828,12,M,Insular,Secundaria,NaN,NaN
1680,0,F,Insular,Primaria,NaN,NaN


In [16]:
duplicados_df = df[
    df.duplicated(keep=False)
]

duplicados_df.head(20)

,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm,alerta_desnutricion,peso_missing,talla_missing,edad_clean,edad_missing
469,-5,F,Sierra,Secundaria,NaN,NaN,0,1,1,NaN,1
769,18,F,Amazonía,Secundaria,NaN,NaN,0,1,1,18.0,0
828,12,M,Insular,Secundaria,NaN,NaN,0,1,1,12.0,0
2090,16,F,Insular,Secundaria,NaN,NaN,0,1,1,16.0,0
2225,6,M,Insular,Superior,NaN,NaN,0,1,1,6.0,0
2936,10,M,Costa,Primaria,NaN,NaN,0,1,1,10.0,0
3193,14,M,Costa,Secundaria,NaN,NaN,0,1,1,14.0,0
3733,4,M,Costa,Primaria,NaN,NaN,0,1,1,4.0,0
3850,6,F,Sierra,Primaria,NaN,NaN,0,1,1,6.0,0
4095,14,F,Sierra,Secundaria,NaN,NaN,0,1,1,14.0,0


###Valores únicos por variable categórica

In [17]:
# ============================================================
# 3.2 VALORES ÚNICOS POR VARIABLE CATEGÓRICA
# ============================================================

# Variables categóricas del dataset

categorical_features = [
    "sexo",
    "region",
    "nivel_educacion_madre"
]

# Recorrer cada variable categórica

for col in categorical_features:

    print("\n" + "="*60)
    print(f"Variable: {col}")
    print("="*60)

    # Número de categorías únicas
    print(
        f"Número de categorías: "
        f"{df[col].nunique()}"
    )

    print("\nCategorías encontradas:")

    print(
        sorted(
            df[col]
            .dropna()
            .unique()
        )
    )


Variable: sexo
Número de categorías: 2

Categorías encontradas:
['F', 'M']

Variable: region
Número de categorías: 4

Categorías encontradas:
['Amazonía', 'Costa', 'Insular', 'Sierra']

Variable: nivel_educacion_madre
Número de categorías: 4

Categorías encontradas:
['Primaria', 'Secundaria', 'Sin escolaridad', 'Superior']


In [18]:
# ============================================================
# RESUMEN DE VARIABLES CATEGÓRICAS
# ============================================================

categorical_summary = pd.DataFrame({

    "Variable": categorical_features,

    "N_Categorias": [
        df[col].nunique()
        for col in categorical_features
    ]

})

display(categorical_summary)

,Variable,N_Categorias
0,sexo,2
1,region,4
2,nivel_educacion_madre,4


###Rango de variables numéricas

In [19]:
# ============================================================
# RANGO DE VARIABLES NUMÉRICAS
# ============================================================

numerical_features = [
    "edad_meses",
    "peso_kg",
    "talla_cm"
]

numeric_ranges = pd.DataFrame({

    "Variable": numerical_features,

    "Minimo": [
        df[col].min()
        for col in numerical_features
    ],

    "Maximo": [
        df[col].max()
        for col in numerical_features
    ]

})

display(numeric_ranges)

,Variable,Minimo,Maximo
0,edad_meses,-12.000000,29.000000
1,peso_kg,-0.814205,59.976104
2,talla_cm,34.904595,98.213648


In [20]:
# ============================================================
# RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS
# ============================================================

numeric_summary = pd.DataFrame({

    "Variable": numerical_features,

    "Minimo": [
        df[col].min()
        for col in numerical_features
    ],

    "Media": [
        df[col].mean()
        for col in numerical_features
    ],

    "Mediana": [
        df[col].median()
        for col in numerical_features
    ],

    "Maximo": [
        df[col].max()
        for col in numerical_features
    ]

})

display(
    numeric_summary.round(2)
)

,Variable,Minimo,Media,Mediana,Maximo
0,edad_meses,-12.00,11.46,11.00,29.00
1,peso_kg,-0.81,6.88,6.52,59.98
2,talla_cm,34.90,67.48,67.38,98.21


###Detección de inconsistencias


Reglas de validación basadas en conocimiento del dominio

Las reglas de plausibilidad fueron definidas tomando como referencia las curvas de crecimiento infantil de la Organización Mundial de la Salud (WHO Child Growth Standards) para niños y niñas de 0 a 2 años. Estas curvas muestran rangos esperados de peso para la edad, longitud para la edad y peso para la longitud.
Las curvas OMS fueron utilizadas como referencia para identificar observaciones potencialmente extremas o biológicamente improbables. Sin embargo, dichas observaciones no fueron consideradas errores de manera automática, ya que podrían corresponder a valores válidos dentro de la variabilidad biológica o a características específicas del proceso de generación sintética de los datos; por lo que se marcaron como observaciones potencialmente inconsistentes o biológicamente improbables para su análisis posterior.
.


In [21]:
# ============================================================
# 3.7 DETECCIÓN DE INCONSISTENCIAS BASADAS EN REGLAS DE DOMINIO
# ============================================================

# Referencia:
# WHO Child Growth Standards para niños y niñas de 0 a 2 años:
# - Weight-for-age
# - Length-for-age
# - Weight-for-length
#
# Estas reglas se usan para revisión inicial.
# No implican eliminación automática de registros.

rules_summary = pd.DataFrame({
    "Variable": [
        "edad_meses",
        "peso_kg",
        "talla_cm"
    ],
    "Regla de plausibilidad": [
        "0 <= edad_meses <= 24",
        "2 <= peso_kg <= 17 según curvas OMS peso-edad 0-2 años",
        "45 <= talla_cm <= 97 según curvas OMS longitud-edad 0-2 años"
    ],
    "Acción inicial": [
        "Marcar como inconsistente si está fuera del rango",
        "Marcar como biológicamente improbable si está fuera del rango",
        "Marcar como biológicamente improbable si está fuera del rango"
    ]
})

display(rules_summary)

,Variable,Regla de plausibilidad,Acción inicial
0,edad_meses,0 <= edad_meses <= 24,Marcar como inconsistente si está fuera del rango
1,peso_kg,2 <= peso_kg <= 17 según curvas OMS peso-edad ...,Marcar como biológicamente improbable si está ...
2,talla_cm,45 <= talla_cm <= 97 según curvas OMS longitud...,Marcar como biológicamente improbable si está ...


In [22]:
# ============================================================
# 3.7.1 CUANTIFICACIÓN DE INCONSISTENCIAS
# ============================================================

edad_menor_0 = (df["edad_meses"] < 0).sum()
edad_mayor_24 = (df["edad_meses"] > 24).sum()

peso_menor_2 = (df["peso_kg"] < 2).sum()
peso_mayor_17 = (df["peso_kg"] > 17).sum()

talla_menor_45 = (df["talla_cm"] < 45).sum()
talla_mayor_97 = (df["talla_cm"] > 97).sum()

inconsistencias_oms = pd.DataFrame({
    "Variable": [
        "edad_meses",
        "edad_meses",
        "peso_kg",
        "peso_kg",
        "talla_cm",
        "talla_cm"
    ],
    "Criterio": [
        "Edad menor a 0 meses",
        "Edad mayor a 24 meses",
        "Peso menor a 2 kg",
        "Peso mayor a 17 kg",
        "Talla menor a 45 cm",
        "Talla mayor a 97 cm"
    ],
    "Casos": [
        edad_menor_0,
        edad_mayor_24,
        peso_menor_2,
        peso_mayor_17,
        talla_menor_45,
        talla_mayor_97
    ]
})

display(inconsistencias_oms)

,Variable,Criterio,Casos
0,edad_meses,Edad menor a 0 meses,3921
1,edad_meses,Edad mayor a 24 meses,3163
2,peso_kg,Peso menor a 2 kg,1383
3,peso_kg,Peso mayor a 17 kg,673
4,talla_cm,Talla menor a 45 cm,514
5,talla_cm,Talla mayor a 97 cm,3


In [23]:
# ============================================================
# 3.7.2 VISUALIZACIÓN DE INCONSISTENCIAS
# ============================================================

fig = px.bar(
    inconsistencias_oms,
    x="Criterio",
    y="Casos",
    color="Variable",
    text="Casos",
    title="Inconsistencias y valores biológicamente improbables según referencia OMS"
)

fig.update_layout(
    xaxis_title="Criterio de validación",
    yaxis_title="Número de registros",
    xaxis_tickangle=-35
)

fig.show()

In [24]:
# ============================================================
# 3.7.3 CREACIÓN DE BANDERAS DE CALIDAD
# ============================================================

df["edad_fuera_rango"] = (
    (df["edad_meses"] < 0) |
    (df["edad_meses"] > 24)
).astype(int)

df["peso_fuera_rango_oms"] = (
    (df["peso_kg"] < 2) |
    (df["peso_kg"] > 17)
).astype(int)

df["talla_fuera_rango_oms"] = (
    (df["talla_cm"] < 45) |
    (df["talla_cm"] > 97)
).astype(int)

quality_flags = df[
    [
        "edad_fuera_rango",
        "peso_fuera_rango_oms",
        "talla_fuera_rango_oms"
    ]
].sum().reset_index()

quality_flags.columns = [
    "Bandera",
    "Casos"
]

display(quality_flags)

,Bandera,Casos
0,edad_fuera_rango,7084
1,peso_fuera_rango_oms,2056
2,talla_fuera_rango_oms,517


In [25]:
# ============================================================
# 3.7.4 RELACIÓN ENTRE BANDERAS DE CALIDAD Y TARGET
# ============================================================

for flag in [
    "edad_fuera_rango",
    "peso_fuera_rango_oms",
    "talla_fuera_rango_oms"
]:

    print("\n" + "="*60)
    print(f"Análisis de {flag}")
    print("="*60)

    display(
        pd.crosstab(
            df[flag],
            df["alerta_desnutricion"],
            normalize="index"
        ).mul(100).round(2)
    )


Análisis de edad_fuera_rango


alerta_desnutricion,0,1
edad_fuera_rango,,
0,89.56,10.44
1,93.79,6.21



Análisis de peso_fuera_rango_oms


alerta_desnutricion,0,1
peso_fuera_rango_oms,,
0,89.9,10.1
1,92.9,7.1



Análisis de talla_fuera_rango_oms


alerta_desnutricion,0,1
talla_fuera_rango_oms,,
0,89.96,10.04
1,93.62,6.38


##4. Análisis Exploratorio de Datos (EDA)

* Análisis Univariante
    * Variable objetivo
    * Variables numéricas
    * Variables categóricas

* Análisis Bivariante
    * Variables numéricas vs target
    * Variables categóricas vs target
    * Variables numéricas vs numéricas

* Análisis Multivariante
    * Correlaciones
    * Missingness Analysis
    * Hallazgos principales





###Análisis Univariante

#### Variable Objetivo

In [26]:
# ============================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================

target_dist = (
    df["alerta_desnutricion"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

target_dist.columns = [
    "Clase",
    "Porcentaje"
]

display(target_dist)

,Clase,Porcentaje
0,0,89.987143
1,1,10.012857


In [27]:
# Distribución de la Variable Objetivo
fig = px.pie(

    target_dist,

    names="Clase",

    values="Porcentaje",

    title="Distribución de la Variable Objetivo"

)

fig.show()

####Variables numéricas

In [28]:
# ============================================================
# 4.1.2 DISTRIBUCIÓN DE EDAD
# ============================================================

fig = px.histogram(
    df,
    x="edad_meses",
    marginal="box",
    nbins=30,
    title="Distribución de Edad (meses)"
)

fig.show()

In [29]:
# ============================================================
# 4.1.2 DISTRIBUCIÓN DE PESO
# ============================================================

fig = px.histogram(
    df,
    x="peso_kg",
    marginal="box",
    nbins=40,
    title="Distribución de Peso (kg)"
)

fig.show()

In [30]:
# ============================================================
# 4.1.2 DISTRIBUCIÓN DE TALLA
# ============================================================

fig = px.histogram(
    df,
    x="talla_cm",
    marginal="box",
    nbins=40,
    title="Distribución de Talla (cm)"
)

fig.show()

In [31]:
# ============================================================
# RESUMEN DE VARIABLES NUMÉRICAS
# ============================================================

df[
    [
        "edad_meses",
        "peso_kg",
        "talla_cm"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
edad_meses,70000.0,11.459457,8.252734,-12.000000,5.000000,11.000000,18.000000,29.000000
peso_kg,66531.0,6.879973,4.648722,-0.814205,4.500895,6.523145,8.548997,59.976104
talla_cm,62951.0,67.480033,11.937076,34.904595,57.442414,67.381530,77.444691,98.213648


####Variables Categóricas

In [32]:
# ============================================================
# DISTRIBUCIÓN DE SEXO
# ============================================================
sexo_freq = (
    df["sexo"]
    .value_counts()
    .reset_index()
)

sexo_freq.columns = [
    "sexo",
    "frecuencia"
]

fig = px.bar(
    sexo_freq,
    x="sexo",
    y="frecuencia",
    text="frecuencia",
    title="Distribución de Sexo"
)

fig.show()

In [33]:
# ============================================================
# DISTRIBUCIÓN DE REGION
# ============================================================
region_freq = (
    df["region"]
    .value_counts()
    .reset_index()
)

region_freq.columns = [
    "region",
    "frecuencia"
]

fig = px.bar(
    region_freq,
    x="region",
    y="frecuencia",
    text="frecuencia",
    title="Distribución por Región"
)

fig.show()

In [34]:
# ============================================================
# DISTRIBUCIÓN DE EDUCACION MADRE
# ============================================================
edu_freq = (
    df["nivel_educacion_madre"]
    .value_counts()
    .reset_index()
)

edu_freq.columns = [
    "nivel_educacion_madre",
    "frecuencia"
]

fig = px.bar(
    edu_freq,
    x="nivel_educacion_madre",
    y="frecuencia",
    text="frecuencia",
    title="Nivel educativo de la madre"
)

fig.show()

###Análisis Bivariante

In [35]:
#Edad vs riesgo
px.box(
    df,
    x="alerta_desnutricion",
    y="edad_meses",
    title="Edad según riesgo de desnutrición"
)

In [36]:
#Peso vs Reisgo
px.box(
    df,
    x="alerta_desnutricion",
    y="peso_kg",
    title="Peso según riesgo de desnutrición"
)

In [37]:
#Talla vs Riesgo
px.box(
    df,
    x="alerta_desnutricion",
    y="talla_cm",
    title="Talla según riesgo de desnutrición"
)

In [38]:
# ============================================================
#  SEXO VS RIESGO DE DESNUTRICIÓN
# ============================================================

sexo_riesgo = (

    pd.crosstab(
        df["sexo"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).reset_index()

sexo_riesgo = sexo_riesgo.rename(
    columns={1: "Riesgo (%)"}
)

fig = px.bar(

    sexo_riesgo,

    x="sexo",

    y="Riesgo (%)",

    text="Riesgo (%)",

    title="Porcentaje de Riesgo de Desnutrición por Sexo"

)

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

In [39]:
# ============================================================
# REGIÓN VS RIESGO DE DESNUTRICIÓN
# ============================================================

region_riesgo = (

    pd.crosstab(
        df["region"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).reset_index()

region_riesgo = region_riesgo.rename(
    columns={1: "Riesgo (%)"}
)

fig = px.bar(

    region_riesgo,

    x="region",

    y="Riesgo (%)",

    text="Riesgo (%)",

    title="Porcentaje de Riesgo de Desnutrición por Región"

)

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

In [40]:
# ============================================================
# EDUCACIÓN DE LA MADRE VS RIESGO
# ============================================================

educacion_riesgo = (

    pd.crosstab(
        df["nivel_educacion_madre"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).reset_index()

educacion_riesgo = educacion_riesgo.rename(
    columns={1: "Riesgo (%)"}
)

fig = px.bar(

    educacion_riesgo,

    x="nivel_educacion_madre",

    y="Riesgo (%)",

    text="Riesgo (%)",

    title="Porcentaje de Riesgo de Desnutrición según Educación de la Madre"

)

fig.update_traces(
    texttemplate='%{text:.2f}%'
)

fig.show()

In [41]:
# Tabla de porcentajes por región

pd.crosstab(
    df["region"],
    df["alerta_desnutricion"],
    normalize="index"
).mul(100).round(2)

alerta_desnutricion,0,1
region,,
Amazonía,98.19,1.81
Costa,87.49,12.51
Insular,87.16,12.84
Sierra,87.04,12.96


In [42]:
# Tabla de porcentajes por región

pd.crosstab(
    df["sexo"],
    df["alerta_desnutricion"],
    normalize="index"
).mul(100).round(2)

alerta_desnutricion,0,1
sexo,,
F,89.96,10.04
M,90.01,9.99


In [43]:
# Tabla de porcentajes por región

pd.crosstab(
    df["nivel_educacion_madre"],
    df["alerta_desnutricion"],
    normalize="index"
).mul(100).round(2)

alerta_desnutricion,0,1
nivel_educacion_madre,,
Primaria,89.41,10.59
Secundaria,89.48,10.52
Sin escolaridad,99.91,0.09
Superior,89.62,10.38


In [44]:
#Edad vs Peso
fig = px.scatter(

    df.sample(5000, random_state=42),

    x="edad_meses",

    y="peso_kg",

    color="alerta_desnutricion",

    opacity=0.5,

    title="Edad vs Peso"

)

fig.show()

In [45]:
#Edad vs Talla
fig = px.scatter(

    df.sample(5000, random_state=42),

    x="edad_meses",

    y="talla_cm",

    color="alerta_desnutricion",

    opacity=0.5,

    title="Edad vs Talla"

)

fig.show()

In [46]:
#Peso vs Talla
fig = px.scatter(

    df.sample(5000, random_state=42),

    x="talla_cm",

    y="peso_kg",

    color="alerta_desnutricion",

    opacity=0.5,

    title="Peso vs Talla"

)

fig.show()

###Análisis Multivariante

In [47]:
# ============================================================
# MATRIZ DE CORRELACIÓN
# ============================================================

# Seleccionar únicamente variables numéricas

corr_matrix = df[
    [
        "edad_meses",
        "peso_kg",
        "talla_cm",
        "alerta_desnutricion"
    ]
].corr()

display(
    corr_matrix.round(3)
)

,edad_meses,peso_kg,talla_cm,alerta_desnutricion
edad_meses,1.000,0.468,0.920,0.028
peso_kg,0.468,1.000,0.463,0.007
talla_cm,0.920,0.463,1.000,0.028
alerta_desnutricion,0.028,0.007,0.028,1.000


In [48]:
# ============================================================
# HEATMAP DE CORRELACIÓN
# ============================================================

fig = px.imshow(

    corr_matrix,

    text_auto=".3f",

    aspect="auto",

    title="Matriz de Correlación"

)

fig.show()

###Análisis de datos faltantes

In [49]:
# ============================================================
# CREACIÓN DE INDICADORES DE VALORES FALTANTES
# ============================================================

df["peso_missing"] = (
    df["peso_kg"]
    .isna()
    .astype(int)
)

df["talla_missing"] = (
    df["talla_cm"]
    .isna()
    .astype(int)
)

In [50]:
# ============================================================
# EDADES FUERA DE RANGO
# ============================================================

df["edad_clean"] = df["edad_meses"]

df.loc[
    (
        df["edad_clean"] < 0
    ) |
    (
        df["edad_clean"] > 24
    ),
    "edad_clean"
] = np.nan

df["edad_missing"] = (
    df["edad_clean"]
    .isna()
    .astype(int)
)

In [51]:
# ============================================================
# PESO FALTANTE VS TARGET
# ============================================================

peso_missing_analysis = (

    pd.crosstab(
        df["peso_missing"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).round(2)

display(
    peso_missing_analysis
)

alerta_desnutricion,0,1
peso_missing,,
0,89.96,10.04
1,90.60,9.40


In [52]:
# ============================================================
# TALLA FALTANTE VS TARGET
# ============================================================

talla_missing_analysis = (

    pd.crosstab(
        df["talla_missing"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).round(2)

display(
    talla_missing_analysis
)

alerta_desnutricion,0,1
talla_missing,,
0,89.97,10.03
1,90.14,9.86


In [53]:
# ============================================================
# EDAD INCONSISTENTE VS TARGET
# ============================================================

edad_missing_analysis = (

    pd.crosstab(
        df["edad_missing"],
        df["alerta_desnutricion"],
        normalize="index"
    )

    * 100

).round(2)

display(
    edad_missing_analysis
)

alerta_desnutricion,0,1
edad_missing,,
0,89.56,10.44
1,93.79,6.21


In [54]:
# ============================================================
# RESUMEN
# ============================================================

missing_summary = pd.DataFrame({

    "Variable": [
        "Edad inválida",
        "Peso faltante",
        "Talla faltante"
    ],

    "Casos": [

        df["edad_missing"].sum(),

        df["peso_missing"].sum(),

        df["talla_missing"].sum()

    ]

})

display(
    missing_summary
)

,Variable,Casos
0,Edad inválida,7084
1,Peso faltante,3469
2,Talla faltante,7049


In [55]:
# ============================================================
# VISUALIZACIÓN DE MISSINGNESS
# ============================================================

fig = px.bar(

    missing_summary,

    x="Variable",

    y="Casos",

    text="Casos",

    title="Valores Faltantes e Inconsistencias Detectadas"

)

fig.show()


###Relación entre banderas de calidad y la variable objetivo

In [56]:
# ============================================================
# RELACIÓN ENTRE BANDERAS DE CALIDAD Y EL TARGET
# ============================================================

quality_flags = [
    "edad_fuera_rango",
    "peso_fuera_rango_oms",
    "talla_fuera_rango_oms"
]

for flag in quality_flags:

    print("\n" + "="*70)
    print(f"Análisis de: {flag}")
    print("="*70)

    tabla = (

        pd.crosstab(
            df[flag],
            df["alerta_desnutricion"],
            normalize="index"
        )

        * 100

    ).round(2)

    display(tabla)


Análisis de: edad_fuera_rango


alerta_desnutricion,0,1
edad_fuera_rango,,
0,89.56,10.44
1,93.79,6.21



Análisis de: peso_fuera_rango_oms


alerta_desnutricion,0,1
peso_fuera_rango_oms,,
0,89.9,10.1
1,92.9,7.1



Análisis de: talla_fuera_rango_oms


alerta_desnutricion,0,1
talla_fuera_rango_oms,,
0,89.96,10.04
1,93.62,6.38


In [57]:
# ============================================================
# RESUMEN DEL RIESGO SEGÚN BANDERAS DE CALIDAD
# ============================================================

flag_summary = []

for flag in [

    "edad_fuera_rango",
    "peso_fuera_rango_oms",
    "talla_fuera_rango_oms"

]:

    riesgo = (

        pd.crosstab(
            df[flag],
            df["alerta_desnutricion"],
            normalize="index"
        )

        * 100

    )

    flag_summary.append({

        "Bandera": flag,

        "Riesgo cuando bandera=0 (%)":
            round(riesgo.loc[0, 1], 2),

        "Riesgo cuando bandera=1 (%)":
            round(riesgo.loc[1, 1], 2)

    })

flag_summary = pd.DataFrame(flag_summary)

display(flag_summary)

,Bandera,Riesgo cuando bandera=0 (%),Riesgo cuando bandera=1 (%)
0,edad_fuera_rango,10.44,6.21
1,peso_fuera_rango_oms,10.10,7.10
2,talla_fuera_rango_oms,10.04,6.38


In [58]:
# ============================================================
# VISUALIZACIÓN DE RIESGO SEGÚN BANDERAS
# ============================================================

flag_plot = flag_summary.melt(

    id_vars="Bandera",

    var_name="Condición",

    value_name="Riesgo (%)"

)

fig = px.bar(

    flag_plot,

    x="Bandera",

    y="Riesgo (%)",

    color="Condición",

    barmode="group",

    text="Riesgo (%)",

    title="Riesgo de Desnutrición según Banderas de Calidad"

)

fig.show()

* Hallazgos principales del EDA
  * La variable objetivo está desbalanceada: clase 0 = 89.99% y clase 1 = 10.01%. Por eso, aunque Kaggle evalúe con Accuracy, en el notebook conviene reportar también Recall, Precision y F1.
  * Sexo no parece aportar señal predictiva relevante: F = 10.04% y M = 9.99%.
  * Región sí parece muy relevante: Amazonía tiene 1.81% de riesgo, mientras Costa, Insular y Sierra están entre 12.51% y 12.96%. Esto probablemente será una variable importante.
  * Educación de la madre muestra un patrón sintético extraño: “Sin escolaridad” tiene solo 0.09% de riesgo, lo cual no debe interpretarse como una relación causal real.
  * Peso, talla y edad individualmente tienen baja correlación con el target: la relación parece depender más de interacciones o variables derivadas.
  * Edad y talla están fuertemente correlacionadas: 0.92, lo cual confirma una relación antropométrica esperada.
  * Los valores faltantes en peso y talla no parecen aportar señal relevante: su riesgo es cercano al promedio general.
  * Las banderas de calidad tienen menor riesgo que los datos normales, lo que sugiere que las anomalías podrían venir del proceso sintético de generación de datos y no representar patrones reales.
  * Las curvas OMS respaldan el uso de reglas de plausibilidad, pero no justifican eliminar automáticamente todos los registros extremos; es mejor marcarlos y tratarlos cuidadosamente. Las curvas OMS usadas incluyen peso-edad, longitud-edad y peso-longitud para niños y niñas de 0 a 2 años

## 5. Preparación de Datos
- Creación de copia de trabajo
- Tratamiento de inconsistencias
- Tratamiento de pesos extremos
- Ingeniería de características
- Preparación para modelado
- Separación de variables numéricas y categóricas
- División Train-Test
- Pipeline de Preprocesamiento
- Imputación
- Codificación categórica
- Escalado (si aplica)
- Comparación de estrategias


###Copia del trabajo

In [59]:
df_model = df.copy()

print(df_model.shape)

(70000, 14)


### Tratamiento de inconsistencias

In [60]:
#Edad
df_model["edad_meses"] = df_model["edad_meses"].where(
    (df_model["edad_meses"] >= 0) &
    (df_model["edad_meses"] <= 24),
    np.nan
)

In [61]:
#Peso
df_model["peso_kg"] = df_model["peso_kg"].where(
    df_model["peso_kg"] > 0,
    np.nan
)

In [62]:
#Talla
df_model["peso_kg"] = df_model["peso_kg"].where(
    df_model["peso_kg"] > 0,
    np.nan
)

In [63]:
print(df_model[["edad_meses","peso_kg","talla_cm"]].isna().sum())

edad_meses    7084
peso_kg       3480
talla_cm      7049
dtype: int64


###Tratamiento de pesos extremos

In [64]:
df_model["peso_extremo"] = (
    df_model["peso_kg"] > 30
).astype(int)

In [65]:
print((df_model["peso_kg"] > 25).sum())

print((df_model["peso_kg"] > 30).sum())

print((df_model["peso_kg"] > 20).sum())

673
673
673


Tratamiento de valores biológicamente implausibles

Durante el análisis exploratorio se identificaron 673 registros (0.96% del total) con pesos superiores a 30 kg en niños menores de 24 meses. La inspección individual mostró valores incompatibles con el crecimiento infantil esperado (por ejemplo, 36–56 kg en lactantes menores de un año), por lo que fueron considerados errores de captura y eliminados del conjunto de datos antes del modelado.

Escenario A: Eliminación

In [66]:
# df_A = df_model.copy()

# df_A = df_A[
#     (df_A["peso_kg"].isna()) |
#     (df_A["peso_kg"] <= 30)
# ].copy()

Escenario B: Corrección por punto decimal

In [67]:
# df_B = df_model.copy()

In [68]:
# mask = df_B["peso_kg"] > 30

# df_B.loc[mask, "peso_kg"] = (
#     df_B.loc[mask, "peso_kg"] / 10
# )

In [69]:
# df_B["peso_corregido"] = 0

# df_B.loc[
#     df_B["peso_kg"] > 30,
#     "peso_corregido"
# ] = 1

In [70]:
# mask = df_B["peso_kg"] > 30

# df_B["peso_corregido"] = mask.astype(int)

# df_B.loc[mask, "peso_kg"] = (
#     df_B.loc[mask, "peso_kg"] / 10
# )

In [71]:
# df_B[
#     df_B["peso_corregido"] == 1
# ][
#     [
#         "edad_meses",
#         "peso_kg",
#         "talla_cm"
#     ]
# ].head(20)

In [72]:
def recalcular_variables(df):

    df["imc"] = (
        df["peso_kg"] /
        (df["talla_cm"]/100)**2
    )

    df["peso_talla_ratio"] = (
        df["peso_kg"] /
        df["talla_cm"]
    )

    df["peso_edad_ratio"] = (
        df["peso_kg"] /
        (df["edad_meses"] + 1)
    )

    df["talla_edad_ratio"] = (
        df["talla_cm"] /
        (df["edad_meses"] + 1)
    )

    return df

###Ingeniería de Características

In [73]:
#IMC (Indice de Masa Corporal
df_model["imc"] = (
    df_model["peso_kg"] /
    (df_model["talla_cm"]/100)**2
)

In [74]:
df_model["peso_talla_ratio"] = (
    df_model["peso_kg"] /
    df_model["talla_cm"]
)

In [75]:
df_model["peso_edad_ratio"] = (
    df_model["peso_kg"] /
    (df_model["edad_meses"] + 1)
)

In [76]:
df_model["talla_edad_ratio"] = (
    df_model["talla_cm"] /
    (df_model["edad_meses"] + 1)
)

In [77]:
df_model[
    [
        "imc",
        "peso_talla_ratio",
        "peso_edad_ratio",
        "talla_edad_ratio"
    ]
].describe()

,imc,peso_talla_ratio,peso_edad_ratio,talla_edad_ratio
count,59868.000000,59868.000000,59801.000000,56582.000000
mean,14.790013,0.099344,0.782243,9.092258
std,11.049046,0.067170,1.287775,9.976069
min,0.059149,0.000334,0.018824,3.041909
25%,12.051636,0.077111,0.437201,4.096142
50%,13.842639,0.096389,0.516013,5.348248
75%,15.725792,0.112068,0.726027,8.900210
max,293.255565,1.318535,59.007033,62.217063


In [78]:
vars_nuevas = [
    "imc",
    "peso_talla_ratio",
    "peso_edad_ratio",
    "talla_edad_ratio"
]

# df_model[vars_nuevas].describe(percentiles=[0.01,0.05,0.95,0.99]).T

In [79]:
for col in vars_nuevas:
    print("\n", "="*50)
    print(col)
    print(df_model[col].quantile([0.001,0.01,0.99,0.999]))


imc
0.001      2.839127
0.010      5.999572
0.990     43.381412
0.999    180.638926
Name: imc, dtype: float64

peso_talla_ratio
0.001    0.014660
0.010    0.031611
0.990    0.369161
0.999    0.948803
Name: peso_talla_ratio, dtype: float64

peso_edad_ratio
0.001     0.263077
0.010     0.325146
0.990     3.979868
0.999    14.786912
Name: peso_edad_ratio, dtype: float64

talla_edad_ratio
0.001     3.201812
0.010     3.353510
0.990    52.351813
0.999    56.955362
Name: talla_edad_ratio, dtype: float64


In [80]:
# IMC extremos

df_model[
    df_model["imc"] > 50
].shape

(579, 19)

In [81]:
df_model[
    df_model["imc"] > 100
].shape

(276, 19)

In [82]:
df_model[
    df_model["imc"] > 150
].shape

(113, 19)

In [83]:
df_model[
    df_model["imc"] > 100
][
    [
        "edad_meses",
        "peso_kg",
        "talla_cm",
        "imc",
        "alerta_desnutricion"
    ]
].head(20)

,edad_meses,peso_kg,talla_cm,imc,alerta_desnutricion
171,4.0,54.269969,56.761818,168.440679,0
761,6.0,35.695411,59.052059,102.362831,0
826,5.0,39.421744,56.268809,124.508896,0
1033,0.0,36.288611,49.712551,146.837926,0
1201,3.0,40.135979,52.225925,147.150445,0
1378,1.0,45.845659,58.527305,133.838535,0
1384,0.0,45.455018,55.066698,149.900735,0
1513,10.0,52.045327,66.515585,117.634555,0
1610,12.0,52.651837,64.894832,125.023975,0
1644,12.0,52.067326,69.654925,107.315293,0


###Preparación para modelado

In [84]:
# Definición de X e y
X = df_model.drop("alerta_desnutricion", axis=1)
y = df_model["alerta_desnutricion"]

In [85]:
# 5.6 División entrenamiento-validación
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [86]:
# 5.7 Identificación de columnas
cat_cols = [
    "sexo",
    "region",
    "nivel_educacion_madre"
]

num_cols = [
    c for c in X_train.columns
    if c not in cat_cols
]

print("Categóricas:", cat_cols)
print("Numéricas:", num_cols)

Categóricas: ['sexo', 'region', 'nivel_educacion_madre']
Numéricas: ['edad_meses', 'peso_kg', 'talla_cm', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']


###Pipeline de Preprocesamiento

In [87]:
#Escenario A
df_A = df_model[df_model["peso_extremo"] == 0].copy()

print(df_A.shape)

(69327, 19)


In [88]:
#Escenario B
df_B = df_model.copy()
mask = df_B["peso_extremo"] == 1
df_B.loc[mask, "peso_kg"] = (
    df_B.loc[mask, "peso_kg"] / 10
)

In [89]:
df_B["imc"] = (
    df_B["peso_kg"] /
    (df_B["talla_cm"]/100)**2
)

In [90]:
df_B["peso_talla_ratio"] = (
    df_B["peso_kg"] /
    df_B["talla_cm"]
)

df_B["peso_edad_ratio"] = (
    df_B["peso_kg"] /
    (df_B["edad_meses"] + 1)
)

df_B["talla_edad_ratio"] = (
    df_B["talla_cm"] /
    (df_B["edad_meses"] + 1)
)

In [91]:
print(df_A.shape)
print(df_B.shape)

(69327, 19)
(70000, 19)


In [92]:
# ============================================================
# 6. FUNCIÓN GENERAL DE PREPARACIÓN PARA MODELADO
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
def preparar_datos_para_modelado(df_input, target_col="alerta_desnutricion"):

    # Separar variables predictoras y variable objetivo
    X = df_input.drop(target_col, axis=1)
    y = df_input[target_col]

    # División estratificada entrenamiento-validación
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # Identificación de columnas categóricas
    cat_cols = [
        "sexo",
        "region",
        "nivel_educacion_madre"
    ]

    # Columnas numéricas
    num_cols = [
        col for col in X_train.columns
        if col not in cat_cols
    ]

    # Transformador para variables numéricas
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]
    )

    # Transformador para variables categóricas
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    # Preprocesador completo
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    return X_train, X_val, y_train, y_val, preprocessor, num_cols, cat_cols

In [93]:
# 5.8 Pipeline de preprocesamiento


# numeric_transformer = Pipeline(
#     steps=[
#         ("imputer", SimpleImputer(strategy="median")),
#         ("scaler", RobustScaler())
#     ]
# )

# categorical_transformer = Pipeline(
#     steps=[
#         ("imputer", SimpleImputer(strategy="most_frequent")),
#         ("onehot", OneHotEncoder(handle_unknown="ignore"))
#     ]
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", numeric_transformer, num_cols),
#         ("cat", categorical_transformer, cat_cols)
#     ]
# )

In [94]:
# 5.9 Ajuste del preprocesador usando solo entrenamiento
# X_train_prepared = preprocessor.fit_transform(X_train)
# X_val_prepared = preprocessor.transform(X_val)

# print(X_train_prepared.shape)
# print(X_val_prepared.shape)

In [95]:
# print("df_model:", df_model.shape)

# try:
#     print("df_A:", df_A.shape)
# except NameError:
#     print("df_A no existe")

# try:
#     print("df_B:", df_B.shape)
# except NameError:
#     print("df_B no existe")

# print("X_train:", X_train.shape)
# print("X_val:", X_val.shape)

In [96]:
# ============================================================
# 6.1 PREPARACIÓN DEL ESCENARIO A
# ============================================================

X_train_A, X_val_A, y_train_A, y_val_A, preprocessor_A, num_cols_A, cat_cols_A = preparar_datos_para_modelado(
    df_A
)

print("X_train_A:", X_train_A.shape)
print("X_val_A:", X_val_A.shape)
print("y_train_A:", y_train_A.shape)
print("y_val_A:", y_val_A.shape)

print("\nDistribución y_train_A:")
print(y_train_A.value_counts(normalize=True))

print("\nDistribución y_val_A:")
print(y_val_A.value_counts(normalize=True))

X_train_A: (55461, 18)
X_val_A: (13866, 18)
y_train_A: (55461,)
y_val_A: (13866,)

Distribución y_train_A:
alerta_desnutricion
0    0.899695
1    0.100305
Name: proportion, dtype: float64

Distribución y_val_A:
alerta_desnutricion
0    0.899683
1    0.100317
Name: proportion, dtype: float64


In [97]:
# ============================================================
# 6.2 PREPARACIÓN DEL ESCENARIO B
# ============================================================

X_train_B, X_val_B, y_train_B, y_val_B, preprocessor_B, num_cols_B, cat_cols_B = preparar_datos_para_modelado(
    df_B
)

print("X_train_B:", X_train_B.shape)
print("X_val_B:", X_val_B.shape)
print("y_train_B:", y_train_B.shape)
print("y_val_B:", y_val_B.shape)

print("\nDistribución y_train_B:")
print(y_train_B.value_counts(normalize=True))

print("\nDistribución y_val_B:")
print(y_val_B.value_counts(normalize=True))

X_train_B: (56000, 18)
X_val_B: (14000, 18)
y_train_B: (56000,)
y_val_B: (14000,)

Distribución y_train_B:
alerta_desnutricion
0    0.899875
1    0.100125
Name: proportion, dtype: float64

Distribución y_val_B:
alerta_desnutricion
0    0.899857
1    0.100143
Name: proportion, dtype: float64


In [98]:
# ============================================================
# 6.3 APLICAR PREPROCESAMIENTO AL ESCENARIO A
# ============================================================

X_train_A_prepared = preprocessor_A.fit_transform(X_train_A)
X_val_A_prepared = preprocessor_A.transform(X_val_A)

print("X_train_A_prepared:", X_train_A_prepared.shape)
print("X_val_A_prepared:", X_val_A_prepared.shape)

X_train_A_prepared: (55461, 25)
X_val_A_prepared: (13866, 25)


In [99]:
# ============================================================
# 6.4 APLICAR PREPROCESAMIENTO AL ESCENARIO B
# ============================================================

X_train_B_prepared = preprocessor_B.fit_transform(X_train_B)
X_val_B_prepared = preprocessor_B.transform(X_val_B)

print("X_train_B_prepared:", X_train_B_prepared.shape)
print("X_val_B_prepared:", X_val_B_prepared.shape)

X_train_B_prepared: (56000, 25)
X_val_B_prepared: (14000, 25)


##6. Modelado baseline

###Función de evaluación

In [100]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

def evaluar_modelo(nombre, modelo, X_val, y_val):

    y_pred = modelo.predict(X_val)

    if hasattr(modelo, "predict_proba"):
        y_proba = modelo.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)
    else:
        auc = None

    resultados = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0),
        "ROC_AUC": auc
    }

    print(f"\n===== {nombre} =====")
    print(classification_report(y_val, y_pred))
    print("Matriz de confusión:")
    print(confusion_matrix(y_val, y_pred))

    return resultados

###DummyClassifier

In [101]:
from sklearn.dummy import DummyClassifier

resultados = []

dummy_A = DummyClassifier(strategy="most_frequent")

dummy_A.fit(X_train_A_prepared, y_train_A)

resultados.append(
    evaluar_modelo(
        "Dummy - Escenario A",
        dummy_A,
        X_val_A_prepared,
        y_val_A
    )
)


===== Dummy - Escenario A =====
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12475
           1       0.00      0.00      0.00      1391

    accuracy                           0.90     13866
   macro avg       0.45      0.50      0.47     13866
weighted avg       0.81      0.90      0.85     13866

Matriz de confusión:
[[12475     0]
 [ 1391     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [102]:
dummy_B = DummyClassifier(strategy="most_frequent")

dummy_B.fit(X_train_B_prepared, y_train_B)

resultados.append(
    evaluar_modelo(
        "Dummy - Escenario B",
        dummy_B,
        X_val_B_prepared,
        y_val_B
    )
)


===== Dummy - Escenario B =====
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12598
           1       0.00      0.00      0.00      1402

    accuracy                           0.90     14000
   macro avg       0.45      0.50      0.47     14000
weighted avg       0.81      0.90      0.85     14000

Matriz de confusión:
[[12598     0]
 [ 1402     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



###Regresión Logística

In [103]:
from sklearn.linear_model import LogisticRegression

logreg_A = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logreg_A.fit(X_train_A_prepared, y_train_A)

resultados.append(
    evaluar_modelo(
        "Logistic Regression - Escenario A",
        logreg_A,
        X_val_A_prepared,
        y_val_A
    )
)


===== Logistic Regression - Escenario A =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12475
           1       0.14      0.94      0.24      1391

    accuracy                           0.42     13866
   macro avg       0.56      0.65      0.38     13866
weighted avg       0.90      0.42      0.50     13866

Matriz de confusión:
[[4484 7991]
 [  89 1302]]


In [104]:
logreg_B = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logreg_B.fit(X_train_B_prepared, y_train_B)

resultados.append(
    evaluar_modelo(
        "Logistic Regression - Escenario B",
        logreg_B,
        X_val_B_prepared,
        y_val_B
    )
)


===== Logistic Regression - Escenario B =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.94      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

Matriz de confusión:
[[4563 8035]
 [  91 1311]]


###Tabla comparativa

In [105]:
resultados_df = pd.DataFrame(resultados)

display(
    resultados_df.sort_values(
        by="F1",
        ascending=False
    )
)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
1,Dummy - Escenario B,0.899857,0.000000,0.000000,0.000000,0.500000
0,Dummy - Escenario A,0.899683,0.000000,0.000000,0.000000,0.500000


###Arbol de decisión

In [106]:
# ==========================================================
# 7.1 DECISION TREE - ESCENARIO A
# ==========================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

dt_A = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

dt_A.fit(X_train_A_prepared, y_train_A)

y_pred_A = dt_A.predict(X_val_A_prepared)
y_prob_A = dt_A.predict_proba(X_val_A_prepared)[:,1]

print("===== Decision Tree - Escenario A =====")
print(classification_report(y_val_A, y_pred_A))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_val_A, y_pred_A))

resultados.append({
    "Modelo": "Decision Tree - Escenario A",
    "Accuracy": accuracy_score(y_val_A, y_pred_A),
    "Precision": precision_score(y_val_A, y_pred_A),
    "Recall": recall_score(y_val_A, y_pred_A),
    "F1": f1_score(y_val_A, y_pred_A),
    "ROC_AUC": roc_auc_score(y_val_A, y_prob_A)
})

===== Decision Tree - Escenario A =====
              precision    recall  f1-score   support

           0       0.98      0.35      0.52     12475
           1       0.14      0.95      0.24      1391

    accuracy                           0.41     13866
   macro avg       0.56      0.65      0.38     13866
weighted avg       0.90      0.41      0.49     13866


Matriz de Confusión:
[[4396 8079]
 [  74 1317]]


In [107]:
# ==========================================================
# 7.2 DECISION TREE - ESCENARIO B
# ==========================================================

dt_B = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

dt_B.fit(X_train_B_prepared, y_train_B)

y_pred_B = dt_B.predict(X_val_B_prepared)
y_prob_B = dt_B.predict_proba(X_val_B_prepared)[:,1]

print("===== Decision Tree - Escenario B =====")
print(classification_report(y_val_B, y_pred_B))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_val_B, y_pred_B))

resultados.append({
    "Modelo": "Decision Tree - Escenario B",
    "Accuracy": accuracy_score(y_val_B, y_pred_B),
    "Precision": precision_score(y_val_B, y_pred_B),
    "Recall": recall_score(y_val_B, y_pred_B),
    "F1": f1_score(y_val_B, y_pred_B),
    "ROC_AUC": roc_auc_score(y_val_B, y_prob_B)
})

===== Decision Tree - Escenario B =====
              precision    recall  f1-score   support

           0       0.98      0.35      0.52     12598
           1       0.14      0.95      0.24      1402

    accuracy                           0.41     14000
   macro avg       0.56      0.65      0.38     14000
weighted avg       0.90      0.41      0.49     14000


Matriz de Confusión:
[[4445 8153]
 [  74 1328]]


In [108]:
pd.DataFrame(resultados).sort_values(
    by="F1",
    ascending=False
)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
1,Dummy - Escenario B,0.899857,0.000000,0.000000,0.000000,0.500000
0,Dummy - Escenario A,0.899683,0.000000,0.000000,0.000000,0.500000


###Random Forest

In [109]:
# ============================================================
# 7.5 RANDOM FOREST - ESCENARIO A
# ============================================================

from sklearn.ensemble import RandomForestClassifier

rf_A = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=20,
    #class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_A.fit(X_train_A_prepared, y_train_A)

resultados.append(
    evaluar_modelo(
        "Random Forest - Escenario A",
        rf_A,
        X_val_A_prepared,
        y_val_A
    )
)


===== Random Forest - Escenario A =====
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12475
           1       0.00      0.00      0.00      1391

    accuracy                           0.90     13866
   macro avg       0.45      0.50      0.47     13866
weighted avg       0.81      0.90      0.85     13866

Matriz de confusión:
[[12475     0]
 [ 1391     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [110]:
# ============================================================
# 7.6 RANDOM FOREST - ESCENARIO B
# ============================================================

rf_B = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=20,
    #class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_B.fit(X_train_B_prepared, y_train_B)

resultados.append(
    evaluar_modelo(
        "Random Forest - Escenario B",
        rf_B,
        X_val_B_prepared,
        y_val_B
    )
)


===== Random Forest - Escenario B =====
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12598
           1       0.00      0.00      0.00      1402

    accuracy                           0.90     14000
   macro avg       0.45      0.50      0.47     14000
weighted avg       0.81      0.90      0.85     14000

Matriz de confusión:
[[12598     0]
 [ 1402     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [111]:
resultados_df = pd.DataFrame(resultados)

display(
    resultados_df.sort_values(
        by="F1",
        ascending=False
    )
)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
0,Dummy - Escenario A,0.899683,0.000000,0.000000,0.000000,0.500000
1,Dummy - Escenario B,0.899857,0.000000,0.000000,0.000000,0.500000
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
7,Random Forest - Escenario B,0.899857,0.000000,0.000000,0.000000,0.647331


In [112]:
# ============================================================
# NOMBRES DE VARIABLES DESPUÉS DEL PREPROCESAMIENTO
# ============================================================

feature_names_A = preprocessor_A.get_feature_names_out()
feature_names_B = preprocessor_B.get_feature_names_out()

In [113]:
# ============================================================
# IMPORTANCIA DE VARIABLES - RANDOM FOREST A
# ============================================================

feature_importance_A = pd.DataFrame({
    "feature": feature_names_A,
    "importance": rf_A.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

display(feature_importance_A.head(20))

,feature,importance
17,cat__region_Amazonía,0.314135
23,cat__nivel_educacion_madre_Sin escolaridad,0.092679
2,num__talla_cm,0.076592
1,num__peso_kg,0.068841
12,num__peso_talla_ratio,0.059077
11,num__imc,0.053424
14,num__talla_edad_ratio,0.052090
13,num__peso_edad_ratio,0.050451
20,cat__region_Sierra,0.043744
19,cat__region_Insular,0.043245


In [114]:
# ============================================================
# IMPORTANCIA DE VARIABLES - RANDOM FOREST B
# ============================================================

feature_importance_B = pd.DataFrame({
    "feature": feature_names_B,
    "importance": rf_B.feature_importances_
}).sort_values(
    by="importance",
    ascending=False
)

display(feature_importance_B.head(20))

,feature,importance
17,cat__region_Amazonía,0.316197
23,cat__nivel_educacion_madre_Sin escolaridad,0.093679
1,num__peso_kg,0.070320
2,num__talla_cm,0.069709
12,num__peso_talla_ratio,0.055739
11,num__imc,0.053062
14,num__talla_edad_ratio,0.050655
13,num__peso_edad_ratio,0.048791
20,cat__region_Sierra,0.047034
19,cat__region_Insular,0.045403


In [115]:
fig = px.bar(
    feature_importance_B.head(15),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 15 Variables más Importantes - Random Forest"
)

fig.show()

In [116]:
y_prob_rf = rf_B.predict_proba(
    X_val_B_prepared
)[:,1]

In [117]:
pd.Series(y_prob_rf).describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99]
)

,0
count,14000.000000
mean,0.099896
std,0.056399
min,0.000407
50%,0.130201
75%,0.139688
90%,0.149169
95%,0.156408
99%,0.176860
max,0.246786


In [118]:
pd.crosstab(
    df_model["region"],
    df_model["alerta_desnutricion"],
    normalize="index"
) * 100

alerta_desnutricion,0,1
region,,
Amazonía,98.188118,1.811882
Costa,87.487168,12.512832
Insular,87.163514,12.836486
Sierra,87.041410,12.958590


In [119]:
pd.crosstab(
    df_model["nivel_educacion_madre"],
    df_model["alerta_desnutricion"],
    normalize="index"
) * 100

alerta_desnutricion,0,1
nivel_educacion_madre,,
Primaria,89.407745,10.592255
Secundaria,89.478532,10.521468
Sin escolaridad,99.912715,0.087285
Superior,89.619674,10.380326


In [120]:
# df_model["nivel_educacion_madre"].value_counts()
df_model["nivel_educacion_madre"].value_counts(dropna=False)

,count
nivel_educacion_madre,
Secundaria,31488
Primaria,24584
Superior,10491
Sin escolaridad,3437


In [121]:
pd.crosstab(
    df_model["nivel_educacion_madre"],
    df_model["alerta_desnutricion"]
)


alerta_desnutricion,0,1
nivel_educacion_madre,,
Primaria,21980,2604
Secundaria,28175,3313
Sin escolaridad,3434,3
Superior,9402,1089


###Balanceo de clases SMOTE

In [122]:
!pip install imbalanced-learn

In [123]:
from imblearn.over_sampling import SMOTE

In [124]:
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)

X_train_A_smote, y_train_A_smote = smote.fit_resample(
    X_train_A_prepared,
    y_train_A
)

print(X_train_A_smote.shape)
print(y_train_A_smote.value_counts())

(99796, 25)
alerta_desnutricion
0    49898
1    49898
Name: count, dtype: int64


In [125]:
smote = SMOTE(
    sampling_strategy='auto',
    random_state=42
)

X_train_B_smote, y_train_B_smote = smote.fit_resample(
    X_train_B_prepared,
    y_train_B
)

print(X_train_B_smote.shape)
print(y_train_B_smote.value_counts())

(100786, 25)
alerta_desnutricion
0    50393
1    50393
Name: count, dtype: int64


In [126]:
y_train_A.value_counts()
y_train_A_smote.value_counts()

y_train_B.value_counts()
y_train_B_smote.value_counts()

,count
alerta_desnutricion,
0,50393
1,50393


###Random Forest + SMOTE

In [127]:
#Escenario A
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

rf_A_smote = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_A_smote.fit(
    X_train_A_smote,
    y_train_A_smote
)

y_pred_rf_A_smote = rf_A_smote.predict(
    X_val_A_prepared
)

y_prob_rf_A_smote = rf_A_smote.predict_proba(
    X_val_A_prepared
)[:,1]

print("\n===== RF + SMOTE - Escenario A =====\n")

print(
    classification_report(
        y_val_A,
        y_pred_rf_A_smote
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_A,
        y_prob_rf_A_smote
    )
)

print(
    confusion_matrix(
        y_val_A,
        y_pred_rf_A_smote
    )
)


===== RF + SMOTE - Escenario A =====

              precision    recall  f1-score   support

           0       0.90      0.90      0.90     12475
           1       0.14      0.14      0.14      1391

    accuracy                           0.83     13866
   macro avg       0.52      0.52      0.52     13866
weighted avg       0.83      0.83      0.83     13866

ROC-AUC: 0.6293840304620744
[[11265  1210]
 [ 1202   189]]


In [128]:
#Escenario B
rf_B_smote = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_B_smote.fit(
    X_train_B_smote,
    y_train_B_smote
)

y_pred_rf_B_smote = rf_B_smote.predict(
    X_val_B_prepared
)

y_prob_rf_B_smote = rf_B_smote.predict_proba(
    X_val_B_prepared
)[:,1]

print("\n===== RF + SMOTE - Escenario B =====\n")

print(
    classification_report(
        y_val_B,
        y_pred_rf_B_smote
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_B,
        y_prob_rf_B_smote
    )
)

print(
    confusion_matrix(
        y_val_B,
        y_pred_rf_B_smote
    )
)


===== RF + SMOTE - Escenario B =====

              precision    recall  f1-score   support

           0       0.91      0.90      0.90     12598
           1       0.15      0.15      0.15      1402

    accuracy                           0.83     14000
   macro avg       0.53      0.53      0.53     14000
weighted avg       0.83      0.83      0.83     14000

ROC-AUC: 0.6298653648123392
[[11347  1251]
 [ 1186   216]]


In [129]:
pd.Series(y_prob_rf_A_smote).describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99]
)

,0
count,13866.000000
mean,0.207468
std,0.209050
min,0.000000
50%,0.150000
75%,0.300000
90%,0.503333
95%,0.666667
99%,0.890000
max,0.996667


In [130]:
pd.Series(y_prob_rf_B_smote).describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99]
)

,0
count,14000.000000
mean,0.205134
std,0.211299
min,0.000000
50%,0.143333
75%,0.296667
90%,0.513333
95%,0.673333
99%,0.886700
max,1.000000


In [131]:
# ============================================================
# AGREGAR RF + SMOTE A LA TABLA COMPARATIVA
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

resultados.append({
    "Modelo": "Random Forest + SMOTE - Escenario A",
    "Accuracy": accuracy_score(y_val_A, y_pred_rf_A_smote),
    "Precision": precision_score(y_val_A, y_pred_rf_A_smote, zero_division=0),
    "Recall": recall_score(y_val_A, y_pred_rf_A_smote, zero_division=0),
    "F1": f1_score(y_val_A, y_pred_rf_A_smote, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val_A, y_prob_rf_A_smote)
})

resultados.append({
    "Modelo": "Random Forest + SMOTE - Escenario B",
    "Accuracy": accuracy_score(y_val_B, y_pred_rf_B_smote),
    "Precision": precision_score(y_val_B, y_pred_rf_B_smote, zero_division=0),
    "Recall": recall_score(y_val_B, y_pred_rf_B_smote, zero_division=0),
    "F1": f1_score(y_val_B, y_pred_rf_B_smote, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val_B, y_prob_rf_B_smote)
})

In [132]:
resultados_df = pd.DataFrame(resultados)

display(
    resultados_df
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(by="F1", ascending=False)
)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
9,Random Forest + SMOTE - Escenario B,0.825929,0.147239,0.154066,0.150575,0.629865
8,Random Forest + SMOTE - Escenario A,0.826049,0.135096,0.135873,0.135484,0.629384
1,Dummy - Escenario B,0.899857,0.000000,0.000000,0.000000,0.500000
0,Dummy - Escenario A,0.899683,0.000000,0.000000,0.000000,0.500000
7,Random Forest - Escenario B,0.899857,0.000000,0.000000,0.000000,0.647331
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499


In [133]:
from sklearn.metrics import f1_score

thresholds = np.arange(0.05, 0.95, 0.01)

best_f1 = 0
best_thr = 0

for thr in thresholds:

    pred = (y_prob_rf_B_smote >= thr).astype(int)

    f1 = f1_score(y_val_B, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print(best_thr, best_f1)

0.09000000000000001 0.22682053322721846


El balanceo mediante SMOTE no mejora el desempeño predictivo del problema. Aunque modifica adecuadamente la distribución de probabilidades y permite ajustar el umbral de decisión, el rendimiento final permanece inferior al obtenido por los modelos entrenados sobre la distribución original utilizando ponderación de clases.

La corrección de los registros extremos de peso mediante desplazamiento decimal no produjo mejoras significativas respecto a la eliminación de dichos registros, indicando que estos representan una fracción muy pequeña del conjunto de datos y tienen un impacto limitado sobre el desempeño predictivo.

###**XGBoost**

In [134]:
!pip install xgboost catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [135]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [136]:
neg = (y_train_B == 0).sum()
pos = (y_train_B == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

8.987515605493133


In [137]:
xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb_model.fit(
    X_train_B_prepared,
    y_train_B
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [138]:
y_pred_xgb = xgb_model.predict(
    X_val_B_prepared
)

y_prob_xgb = xgb_model.predict_proba(
    X_val_B_prepared
)[:,1]

In [139]:
print("\n===== XGBoost - Escenario B =====\n")

print(
    classification_report(
        y_val_B,
        y_pred_xgb
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_B,
        y_prob_xgb
    )
)

print(
    confusion_matrix(
        y_val_B,
        y_pred_xgb
    )
)


===== XGBoost - Escenario B =====

              precision    recall  f1-score   support

           0       0.94      0.50      0.65     12598
           1       0.14      0.73      0.23      1402

    accuracy                           0.52     14000
   macro avg       0.54      0.62      0.44     14000
weighted avg       0.86      0.52      0.61     14000

ROC-AUC: 0.6517856071169507
[[6263 6335]
 [ 374 1028]]


In [140]:
resultados.append({
    "Modelo": "XGBoost - Escenario B",
    "Accuracy": accuracy_score(y_val_B, y_pred_xgb),
    "Precision": precision_score(y_val_B, y_pred_xgb),
    "Recall": recall_score(y_val_B, y_pred_xgb),
    "F1": f1_score(y_val_B, y_pred_xgb),
    "ROC_AUC": roc_auc_score(y_val_B, y_prob_xgb)
})

resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values("F1", ascending=False)
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
9,Random Forest + SMOTE - Escenario B,0.825929,0.147239,0.154066,0.150575,0.629865
8,Random Forest + SMOTE - Escenario A,0.826049,0.135096,0.135873,0.135484,0.629384
1,Dummy - Escenario B,0.899857,0.000000,0.000000,0.000000,0.500000
0,Dummy - Escenario A,0.899683,0.000000,0.000000,0.000000,0.500000
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499


In [141]:
pd.Series(y_prob_xgb).describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99]
)

,0
count,14000.000000
mean,0.407615
std,0.224868
min,0.000450
50%,0.511472
75%,0.586991
90%,0.631467
95%,0.658900
99%,0.711698
max,0.876561


In [142]:
feature_names = preprocessor_B.get_feature_names_out()

feat_imp_xgb = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_
})

feat_imp_xgb = (
    feat_imp_xgb
    .sort_values("importance", ascending=False)
)

display(feat_imp_xgb.head(20))

,feature,importance
17,cat__region_Amazonía,0.264776
23,cat__nivel_educacion_madre_Sin escolaridad,0.220553
6,num__edad_missing,0.089802
18,cat__region_Costa,0.046465
7,num__edad_fuera_rango,0.046439
19,cat__region_Insular,0.027341
20,cat__region_Sierra,0.025041
4,num__talla_missing,0.022142
2,num__talla_cm,0.019887
24,cat__nivel_educacion_madre_Superior,0.018928


###CatBoost

In [143]:
from catboost import CatBoostClassifier

print("CatBoost OK")

CatBoost OK


In [144]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

cat_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=False,
    random_state=42
)

cat_model.fit(
    X_train_B_prepared,
    y_train_B
)

y_pred_cat = cat_model.predict(X_val_B_prepared)

y_prob_cat = cat_model.predict_proba(
    X_val_B_prepared
)[:,1]

In [145]:
print("\n===== CatBoost - Escenario B =====\n")

print(
    classification_report(
        y_val_B,
        y_pred_cat
    )
)

print(
    confusion_matrix(
        y_val_B,
        y_pred_cat
    )
)

roc_auc_cat = roc_auc_score(
    y_val_B,
    y_prob_cat
)

print("\nROC-AUC:", roc_auc_cat)


===== CatBoost - Escenario B =====

              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12598
           1       0.00      0.00      0.00      1402

    accuracy                           0.90     14000
   macro avg       0.45      0.50      0.47     14000
weighted avg       0.81      0.90      0.85     14000

[[12598     0]
 [ 1402     0]]

ROC-AUC: 0.6513686761411079


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [146]:
pd.Series(y_prob_cat).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

,0
count,14000.000000
mean,0.099671
std,0.058896
min,0.000099
50%,0.127815
75%,0.142572
90%,0.154575
95%,0.163840
99%,0.193376
max,0.343300


In [147]:
feature_names = preprocessor_B.get_feature_names_out()

importance_df_cat = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df_cat = (
    importance_df_cat
    .sort_values(
        "importance",
        ascending=False
    )
)

importance_df_cat.head(20)

,feature,importance
17,cat__region_Amazonía,31.841368
23,cat__nivel_educacion_madre_Sin escolaridad,29.599529
1,num__peso_kg,5.504043
2,num__talla_cm,4.810145
14,num__talla_edad_ratio,3.304829
13,num__peso_edad_ratio,2.983013
11,num__imc,2.916665
0,num__edad_meses,2.864331
12,num__peso_talla_ratio,2.662332
5,num__edad_clean,2.509998


In [148]:
resultados.append({
    "Modelo": "CatBoost - Escenario B",
    "Accuracy": accuracy_score(y_val_B, y_pred_cat),
    "Precision": precision_score(y_val_B, y_pred_cat),
    "Recall": recall_score(y_val_B, y_pred_cat),
    "F1": f1_score(y_val_B, y_pred_cat),
    "ROC_AUC": roc_auc_cat
})

pd.DataFrame(resultados).sort_values(
    "ROC_AUC",
    ascending=False
)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.



,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
11,CatBoost - Escenario B,0.899857,0.000000,0.000000,0.000000,0.651369
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
7,Random Forest - Escenario B,0.899857,0.000000,0.000000,0.000000,0.647331
9,Random Forest + SMOTE - Escenario B,0.825929,0.147239,0.154066,0.150575,0.629865
8,Random Forest + SMOTE - Escenario A,0.826049,0.135096,0.135873,0.135484,0.629384


###Experimento 1: eliminar Región

In [149]:
X_train_no_region = X_train_B.drop(
    columns=["region"]
)

X_val_no_region = X_val_B.drop(
    columns=["region"]
)

###Experimento 2: solo variables numericas

In [150]:
numeric_cols_only = [
    "edad_meses",
    "peso_kg",
    "talla_cm",
    "imc",
    "peso_talla_ratio",
    "peso_edad_ratio",
    "talla_edad_ratio",
    "peso_missing",
    "talla_missing",
    "edad_missing",
    "edad_fuera_rango",
    "edad_clean",
    "peso_fuera_rango_oms",
    "talla_fuera_rango_oms",
    "peso_corregido"
]

###Experimentos

In [151]:
# ============================================================
# 8.1 FUNCIÓN AUXILIAR PARA ENTRENAR Y EVALUAR EXPERIMENTOS
# ============================================================

def entrenar_evaluar_experimento(
    nombre_experimento,
    df_input,
    columnas_excluir=None,
    usar_solo_columnas=None
):
    """
    Entrena una regresión logística balanceada sobre un subconjunto
    de variables y evalúa el desempeño en validación.
    """

    if columnas_excluir is None:
        columnas_excluir = []

    # Copia de trabajo
    df_exp = df_input.copy()

    # Definir columnas a usar
    if usar_solo_columnas is not None:
        columnas_modelo = usar_solo_columnas + ["alerta_desnutricion"]
        df_exp = df_exp[columnas_modelo]
    else:
        df_exp = df_exp.drop(columns=columnas_excluir)

    # Separar X e y
    X = df_exp.drop("alerta_desnutricion", axis=1)
    y = df_exp["alerta_desnutricion"]

    # Split estratificado
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # Identificar categóricas presentes
    cat_cols = [
        col for col in [
            "sexo",
            "region",
            "nivel_educacion_madre"
        ]
        if col in X_train.columns
    ]

    # Identificar numéricas
    num_cols = [
        col for col in X_train.columns
        if col not in cat_cols
    ]

    # Pipeline numérico
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]
    )

    # Pipeline categórico
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    # Preprocesador
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols)
        ]
    )

    # Aplicar preprocesamiento
    X_train_prepared = preprocessor.fit_transform(X_train)
    X_val_prepared = preprocessor.transform(X_val)

    # Modelo base
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_prepared, y_train)

    # Predicciones
    y_pred = model.predict(X_val_prepared)
    y_prob = model.predict_proba(X_val_prepared)[:, 1]

    # Resultados
    resultado = {
        "Modelo": nombre_experimento,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred, zero_division=0),
        "Recall": recall_score(y_val, y_pred, zero_division=0),
        "F1": f1_score(y_val, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_val, y_prob)
    }

    print(f"\n===== {nombre_experimento} =====")
    print(classification_report(y_val, y_pred))
    print("Matriz de confusión:")
    print(confusion_matrix(y_val, y_pred))

    return resultado

In [152]:
resultado_completo = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Modelo completo",
    df_input=df_B
)


===== Ablation - Modelo completo =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.94      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

Matriz de confusión:
[[4563 8035]
 [  91 1311]]


In [153]:
resultado_sin_region = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Sin región",
    df_input=df_B,
    columnas_excluir=["region"]
)


===== Ablation - Sin región =====
              precision    recall  f1-score   support

           0       0.93      0.28      0.43     12598
           1       0.11      0.81      0.20      1402

    accuracy                           0.34     14000
   macro avg       0.52      0.54      0.31     14000
weighted avg       0.85      0.34      0.41     14000

Matriz de confusión:
[[3559 9039]
 [ 270 1132]]


In [154]:
num_cols_only = [
    col for col in df_B.columns
    if col not in [
        "alerta_desnutricion",
        "sexo",
        "region",
        "nivel_educacion_madre"
    ]
]

print(num_cols_only)

['edad_meses', 'peso_kg', 'talla_cm', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']


In [155]:
resultado_solo_numericas = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Solo variables numéricas",
    df_input=df_B,
    usar_solo_columnas=num_cols_only
)


===== Ablation - Solo variables numéricas =====
              precision    recall  f1-score   support

           0       0.91      0.38      0.54     12598
           1       0.11      0.67      0.19      1402

    accuracy                           0.41     14000
   macro avg       0.51      0.53      0.36     14000
weighted avg       0.83      0.41      0.50     14000

Matriz de confusión:
[[4788 7810]
 [ 462  940]]


In [156]:
ablation_results = pd.DataFrame([
    resultado_completo,
    resultado_sin_region,
    resultado_solo_numericas
])

display(
    ablation_results.sort_values(
        by="ROC_AUC",
        ascending=False
    )
)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
0,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
1,Ablation - Sin región,0.335071,0.111297,0.807418,0.195628,0.557362
2,Ablation - Solo variables numéricas,0.409143,0.107429,0.670471,0.185185,0.532549


In [157]:
resultados.extend([
    resultado_completo,
    resultado_sin_region,
    resultado_solo_numericas
])

resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(by="ROC_AUC", ascending=False)
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
11,CatBoost - Escenario B,0.899857,0.000000,0.000000,0.000000,0.651369
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
7,Random Forest - Escenario B,0.899857,0.000000,0.000000,0.000000,0.647331
9,Random Forest + SMOTE - Escenario B,0.825929,0.147239,0.154066,0.150575,0.629865


In [158]:
resultado_sin_educacion = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Sin educacion madre",
    df_input=df_B,
    columnas_excluir=[
        "nivel_educacion_madre"
    ]
)


===== Ablation - Sin educacion madre =====
              precision    recall  f1-score   support

           0       0.98      0.33      0.50     12598
           1       0.13      0.92      0.23      1402

    accuracy                           0.39     14000
   macro avg       0.55      0.63      0.36     14000
weighted avg       0.89      0.39      0.47     14000

Matriz de confusión:
[[4181 8417]
 [ 107 1295]]


In [159]:
resultado_sin_sexo = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Sin sexo",
    df_input=df_B,
    columnas_excluir=[
        "sexo"
    ]
)


===== Ablation - Sin sexo =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.93      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

Matriz de confusión:
[[4565 8033]
 [  94 1308]]


In [160]:
resultado_solo_region = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Solo region",
    df_input=df_B,
    usar_solo_columnas=[
        "region"
    ]
)


===== Ablation - Solo region =====
              precision    recall  f1-score   support

           0       0.98      0.28      0.43     12598
           1       0.13      0.95      0.23      1402

    accuracy                           0.35     14000
   macro avg       0.55      0.62      0.33     14000
weighted avg       0.90      0.35      0.41     14000

Matriz de confusión:
[[3499 9099]
 [  66 1336]]


In [161]:
resultado_solo_educacion = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Solo educacion madre",
    df_input=df_B,
    usar_solo_columnas=[
        "nivel_educacion_madre"
    ]
)


===== Ablation - Solo educacion madre =====
              precision    recall  f1-score   support

           0       1.00      0.05      0.10     12598
           1       0.11      1.00      0.19      1402

    accuracy                           0.15     14000
   macro avg       0.55      0.53      0.14     14000
weighted avg       0.91      0.15      0.11     14000

Matriz de confusión:
[[  659 11939]
 [    0  1402]]


In [162]:
resultado_region_educacion = entrenar_evaluar_experimento(
    nombre_experimento="Ablation - Region + Educacion",
    df_input=df_B,
    usar_solo_columnas=[
        "region",
        "nivel_educacion_madre"
    ]
)


===== Ablation - Region + Educacion =====
              precision    recall  f1-score   support

           0       0.98      0.32      0.48     12598
           1       0.13      0.95      0.24      1402

    accuracy                           0.38     14000
   macro avg       0.56      0.63      0.36     14000
weighted avg       0.90      0.38      0.45     14000

Matriz de confusión:
[[3987 8611]
 [  66 1336]]


In [163]:
resultados.extend([
    resultado_sin_educacion,
    resultado_sin_sexo,
    resultado_solo_region,
    resultado_solo_educacion,
    resultado_region_educacion
])

resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(
        by="ROC_AUC",
        ascending=False
    )
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
11,CatBoost - Escenario B,0.899857,0.000000,0.000000,0.000000,0.651369
4,Decision Tree - Escenario A,0.412015,0.140166,0.946801,0.244183,0.647577
7,Random Forest - Escenario B,0.899857,0.000000,0.000000,0.000000,0.647331


In [165]:
# ============================================================
# 9. ESCENARIO B SIN SEXO
# ============================================================

df_B_no_sexo = df_B.drop(
    columns=["sexo"]
).copy()

print(df_B_no_sexo.shape)
print(df_B_no_sexo.columns.tolist())

(70000, 18)
['edad_meses', 'region', 'nivel_educacion_madre', 'peso_kg', 'talla_cm', 'alerta_desnutricion', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']


In [166]:
# ============================================================
# 9.1 SEPARACIÓN X / y
# ============================================================

X_no_sexo = df_B_no_sexo.drop(
    "alerta_desnutricion",
    axis=1
)

y_no_sexo = df_B_no_sexo["alerta_desnutricion"]

In [167]:
# ============================================================
# 9.2 DIVISIÓN TRAIN / VALIDATION
# ============================================================

X_train_no_sexo, X_val_no_sexo, y_train_no_sexo, y_val_no_sexo = train_test_split(
    X_no_sexo,
    y_no_sexo,
    test_size=0.20,
    stratify=y_no_sexo,
    random_state=42
)

print(y_train_no_sexo.value_counts(normalize=True))
print(y_val_no_sexo.value_counts(normalize=True))

alerta_desnutricion
0    0.899875
1    0.100125
Name: proportion, dtype: float64
alerta_desnutricion
0    0.899857
1    0.100143
Name: proportion, dtype: float64


In [168]:
# ============================================================
# 9.3 IDENTIFICACIÓN DE COLUMNAS
# ============================================================

cat_cols_no_sexo = [
    "region",
    "nivel_educacion_madre"
]

num_cols_no_sexo = [
    c for c in X_train_no_sexo.columns
    if c not in cat_cols_no_sexo
]

print("Categóricas:", cat_cols_no_sexo)
print("Numéricas:", num_cols_no_sexo)

Categóricas: ['region', 'nivel_educacion_madre']
Numéricas: ['edad_meses', 'peso_kg', 'talla_cm', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']


In [169]:
# ============================================================
# 9.4 PIPELINE DE PREPROCESAMIENTO SIN SEXO
# ============================================================

numeric_transformer_no_sexo = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]
)

categorical_transformer_no_sexo = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_no_sexo = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_no_sexo, num_cols_no_sexo),
        ("cat", categorical_transformer_no_sexo, cat_cols_no_sexo)
    ]
)

In [170]:
# ============================================================
# 9.5 APLICAR PREPROCESAMIENTO
# ============================================================

X_train_no_sexo_prepared = preprocessor_no_sexo.fit_transform(
    X_train_no_sexo
)

X_val_no_sexo_prepared = preprocessor_no_sexo.transform(
    X_val_no_sexo
)

print(X_train_no_sexo_prepared.shape)
print(X_val_no_sexo_prepared.shape)

(56000, 23)
(14000, 23)


In [171]:
# ============================================================
# 9.6 LOGISTIC REGRESSION - SIN SEXO
# ============================================================

logreg_no_sexo = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logreg_no_sexo.fit(
    X_train_no_sexo_prepared,
    y_train_no_sexo
)

y_pred_logreg_no_sexo = logreg_no_sexo.predict(
    X_val_no_sexo_prepared
)

y_prob_logreg_no_sexo = logreg_no_sexo.predict_proba(
    X_val_no_sexo_prepared
)[:, 1]

print("===== Logistic Regression - Escenario B sin sexo =====")
print(classification_report(y_val_no_sexo, y_pred_logreg_no_sexo))
print(confusion_matrix(y_val_no_sexo, y_pred_logreg_no_sexo))
print("ROC-AUC:", roc_auc_score(y_val_no_sexo, y_prob_logreg_no_sexo))

===== Logistic Regression - Escenario B sin sexo =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.93      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

[[4565 8033]
 [  94 1308]]
ROC-AUC: 0.658859703972213


In [172]:
resultados.append({
    "Modelo": "Logistic Regression - Escenario B sin sexo",
    "Accuracy": accuracy_score(y_val_no_sexo, y_pred_logreg_no_sexo),
    "Precision": precision_score(y_val_no_sexo, y_pred_logreg_no_sexo, zero_division=0),
    "Recall": recall_score(y_val_no_sexo, y_pred_logreg_no_sexo, zero_division=0),
    "F1": f1_score(y_val_no_sexo, y_pred_logreg_no_sexo, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val_no_sexo, y_prob_logreg_no_sexo)
})

In [173]:
# ============================================================
# 9.7 XGBOOST - SIN SEXO
# ============================================================

neg_no_sexo = (y_train_no_sexo == 0).sum()
pos_no_sexo = (y_train_no_sexo == 1).sum()

scale_pos_weight_no_sexo = neg_no_sexo / pos_no_sexo

print("scale_pos_weight:", scale_pos_weight_no_sexo)

scale_pos_weight: 8.987515605493133


In [174]:
xgb_no_sexo = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight_no_sexo,
    random_state=42
)

xgb_no_sexo.fit(
    X_train_no_sexo_prepared,
    y_train_no_sexo
)

y_pred_xgb_no_sexo = xgb_no_sexo.predict(
    X_val_no_sexo_prepared
)

y_prob_xgb_no_sexo = xgb_no_sexo.predict_proba(
    X_val_no_sexo_prepared
)[:, 1]

print("===== XGBoost - Escenario B sin sexo =====")
print(classification_report(y_val_no_sexo, y_pred_xgb_no_sexo))
print(confusion_matrix(y_val_no_sexo, y_pred_xgb_no_sexo))
print("ROC-AUC:", roc_auc_score(y_val_no_sexo, y_prob_xgb_no_sexo))

===== XGBoost - Escenario B sin sexo =====
              precision    recall  f1-score   support

           0       0.95      0.49      0.64     12598
           1       0.14      0.75      0.24      1402

    accuracy                           0.51     14000
   macro avg       0.54      0.62      0.44     14000
weighted avg       0.87      0.51      0.60     14000

[[6159 6439]
 [ 353 1049]]
ROC-AUC: 0.6515921169472138


In [175]:
resultados.append({
    "Modelo": "XGBoost - Escenario B sin sexo",
    "Accuracy": accuracy_score(y_val_no_sexo, y_pred_xgb_no_sexo),
    "Precision": precision_score(y_val_no_sexo, y_pred_xgb_no_sexo, zero_division=0),
    "Recall": recall_score(y_val_no_sexo, y_pred_xgb_no_sexo, zero_division=0),
    "F1": f1_score(y_val_no_sexo, y_pred_xgb_no_sexo, zero_division=0),
    "ROC_AUC": roc_auc_score(y_val_no_sexo, y_prob_xgb_no_sexo)
})

In [176]:
resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(by="ROC_AUC", ascending=False)
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
20,Logistic Regression - Escenario B sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
21,XGBoost - Escenario B sin sexo,0.514857,0.140091,0.748217,0.235996,0.651592
11,CatBoost - Escenario B,0.899857,0.000000,0.000000,0.000000,0.651369


###Optimización LR

In [177]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [178]:
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"]
}

In [179]:
lr_grid = GridSearchCV(
    estimator=LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ),
    param_grid=param_grid_lr,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(
    X_train_no_sexo_prepared,
    y_train_no_sexo
)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=2000, random_state=42),
             n_jobs=-1,
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
             scoring='roc_auc', verbose=1)

In [180]:
print("Mejores parámetros:")
print(lr_grid.best_params_)

print("\nMejor ROC-AUC CV:")
print(lr_grid.best_score_)

Mejores parámetros:
{'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}

Mejor ROC-AUC CV:
0.6521436168646656


In [181]:
best_lr = lr_grid.best_estimator_

y_prob_best_lr = best_lr.predict_proba(
    X_val_no_sexo_prepared
)[:,1]

y_pred_best_lr = best_lr.predict(
    X_val_no_sexo_prepared
)

print(classification_report(
    y_val_no_sexo,
    y_pred_best_lr
))

print(confusion_matrix(
    y_val_no_sexo,
    y_pred_best_lr
))

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_no_sexo,
        y_prob_best_lr
    )
)

              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.94      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

[[4544 8054]
 [  91 1311]]
ROC-AUC: 0.656248393479571


In [183]:
resultados.append({
    "Modelo": "LR Optimizada",
    "Accuracy": accuracy_score(
        y_val_no_sexo,
        y_pred_best_lr
    ),
    "Precision": precision_score(
        y_val_no_sexo,
        y_pred_best_lr,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_no_sexo,
        y_pred_best_lr,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_no_sexo,
        y_pred_best_lr,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_val_no_sexo,
        y_prob_best_lr
    )
})

In [184]:
resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(
        by="ROC_AUC",
        ascending=False
    )
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
20,Logistic Regression - Escenario B sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
23,LR Optimizada,0.418214,0.139989,0.935093,0.243522,0.656248
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786
21,XGBoost - Escenario B sin sexo,0.514857,0.140091,0.748217,0.235996,0.651592


###Optimización de XGBoost

In [185]:
param_grid_xgb = {
    "n_estimators": [200, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

In [186]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

In [187]:
scale_pos_weight = (
    (y_train_no_sexo == 0).sum()
    /
    (y_train_no_sexo == 1).sum()
)

print(scale_pos_weight)

8.987515605493133


In [188]:
param_grid_xgb = {
    "n_estimators": [200, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1]
}

In [189]:
xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=scale_pos_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),
    param_grid=param_grid_xgb,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(
    X_train_no_sexo_prepared,
    y_train_no_sexo
)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=0.8, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='auc', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=Non...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 5, 7], 'n_estimators': [200, 500]},
             scoring='roc_auc', verbose=1)

In [192]:
print("Mejores parámetros:")
print(xgb_grid.best_params_)

print("\nMejor ROC-AUC CV:")
print(xgb_grid.best_score_)

Mejores parámetros:
{'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}

Mejor ROC-AUC CV:
0.6534607518636839


In [191]:
X_val_no_sexo_prepared

array([[ 0.        ,  0.99752419,  0.82078386, ...,  1.        ,
         0.        ,  0.        ],
       [-0.54545455,  0.14102501,  0.        , ...,  1.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.75623103, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.36363636,  0.28507402,  0.67580415, ...,  1.        ,
         0.        ,  0.        ],
       [-0.36363636, -0.03484429,  0.        , ...,  1.        ,
         0.        ,  0.        ],
       [ 1.        ,  1.23555334,  0.90872122, ...,  1.        ,
         0.        ,  0.        ]])

In [193]:
best_xgb = xgb_grid.best_estimator_

y_prob_best_xgb = best_xgb.predict_proba(
    X_val_no_sexo_prepared
)[:,1]

y_pred_best_xgb = best_xgb.predict(
    X_val_no_sexo_prepared
)

print(classification_report(
    y_val_no_sexo,
    y_pred_best_xgb
))

print(confusion_matrix(
    y_val_no_sexo,
    y_pred_best_xgb
))

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val_no_sexo,
        y_prob_best_xgb
    )
)

              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.94      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.38     14000
weighted avg       0.90      0.42      0.50     14000

[[4521 8077]
 [  86 1316]]
ROC-AUC: 0.6528241411867337


In [194]:
resultados.append({
    "Modelo": "XGBoost Optimizado",
    "Accuracy": accuracy_score(
        y_val_no_sexo,
        y_pred_best_xgb
    ),
    "Precision": precision_score(
        y_val_no_sexo,
        y_pred_best_xgb,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_no_sexo,
        y_pred_best_xgb,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_no_sexo,
        y_pred_best_xgb,
        zero_division=0
    ),
    "ROC_AUC": roc_auc_score(
        y_val_no_sexo,
        y_prob_best_xgb
    )
})

In [195]:
resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(
        by="ROC_AUC",
        ascending=False
    )
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
20,Logistic Regression - Escenario B sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
23,LR Optimizada,0.418214,0.139989,0.935093,0.243522,0.656248
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
24,XGBoost Optimizado,0.416929,0.140104,0.938659,0.243817,0.652824
10,XGBoost - Escenario B,0.520786,0.139617,0.733238,0.234569,0.651786


###Undersapling + LR

In [196]:
from imblearn.under_sampling import RandomUnderSampler

In [197]:
# ==================================================
# UNDERSAMPLING
# ==================================================

from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(
    sampling_strategy=1.0,
    random_state=42
)

X_train_B_under, y_train_B_under = rus.fit_resample(
    X_train_B_prepared,
    y_train_B
)

print(y_train_B.value_counts())
print()
print(y_train_B_under.value_counts())

alerta_desnutricion
0    50393
1     5607
Name: count, dtype: int64

alerta_desnutricion
0    5607
1    5607
Name: count, dtype: int64


In [198]:
from sklearn.linear_model import LogisticRegression

lr_under = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_under.fit(
    X_train_B_under,
    y_train_B_under
)

y_pred_lr_under = lr_under.predict(X_val_B_prepared)
y_prob_lr_under = lr_under.predict_proba(X_val_B_prepared)[:,1]

In [199]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print(classification_report(
    y_val_B,
    y_pred_lr_under
))

print(confusion_matrix(
    y_val_B,
    y_pred_lr_under
))

print(
    "ROC-AUC:",
    roc_auc_score(y_val_B, y_prob_lr_under)
)

              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.93      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.39     14000
weighted avg       0.90      0.42      0.50     14000

[[4584 8014]
 [  98 1304]]
ROC-AUC: 0.6577635333280942


In [200]:
resultados.append({
    "Modelo": "LR + UnderSampling",
    "Accuracy": accuracy_score(y_val_B, y_pred_lr_under),
    "Precision": precision_score(y_val_B, y_pred_lr_under),
    "Recall": recall_score(y_val_B, y_pred_lr_under),
    "F1": f1_score(y_val_B, y_pred_lr_under),
    "ROC_AUC": roc_auc_score(y_val_B, y_prob_lr_under)
})

In [202]:
resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(
        by="ROC_AUC",
        ascending=False
    )
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
20,Logistic Regression - Escenario B sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
25,LR + UnderSampling,0.420571,0.139944,0.930100,0.243284,0.657764
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
23,LR Optimizada,0.418214,0.139989,0.935093,0.243522,0.656248
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076
24,XGBoost Optimizado,0.416929,0.140104,0.938659,0.243817,0.652824


###Experimento final de ingeniería de características

In [205]:
# ============================================================
# EXPERIMENTO FINAL: LOGISTIC REGRESSION + POLYNOMIAL FEATURES
# Escenario B sin sexo
# ============================================================

from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Variables numéricas sobre las que queremos generar interacciones
poly_cols = [
    "edad_meses",
    "peso_kg",
    "talla_cm",
    "imc",
    "peso_talla_ratio",
    "peso_edad_ratio",
    "talla_edad_ratio"
]

# Variables categóricas restantes
cat_cols_poly = [
    "region",
    "nivel_educacion_madre"
]

# Otras variables numéricas que NO queremos combinar polinómicamente
other_num_cols = [
    col for col in X_train_no_sexo.columns
    if col not in poly_cols + cat_cols_poly
]

print("Polynomial cols:", poly_cols)
print("Other numeric cols:", other_num_cols)
print("Categorical cols:", cat_cols_poly)

Polynomial cols: ['edad_meses', 'peso_kg', 'talla_cm', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']
Other numeric cols: ['peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo']
Categorical cols: ['region', 'nivel_educacion_madre']


In [206]:
# Pipeline para variables polinómicas
poly_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", RobustScaler())
    ]
)

# Pipeline para otras numéricas
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]
)

# Pipeline para categóricas
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Preprocesador completo
preprocessor_poly = ColumnTransformer(
    transformers=[
        ("poly", poly_transformer, poly_cols),
        ("num", numeric_transformer, other_num_cols),
        ("cat", categorical_transformer, cat_cols_poly)
    ]
)

In [207]:
# Modelo
lr_poly = Pipeline(
    steps=[
        ("preprocessor", preprocessor_poly),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

# Entrenamiento
lr_poly.fit(
    X_train_no_sexo,
    y_train_no_sexo
)

# Predicciones
y_pred_lr_poly = lr_poly.predict(X_val_no_sexo)

y_prob_lr_poly = lr_poly.predict_proba(X_val_no_sexo)[:, 1]

In [208]:
# Evaluación
print("===== Logistic Regression + Polynomial Features =====")

print(classification_report(
    y_val_no_sexo,
    y_pred_lr_poly
))

print(confusion_matrix(
    y_val_no_sexo,
    y_pred_lr_poly
))

roc_lr_poly = roc_auc_score(
    y_val_no_sexo,
    y_prob_lr_poly
)

print("ROC-AUC:", roc_lr_poly)

===== Logistic Regression + Polynomial Features =====
              precision    recall  f1-score   support

           0       0.98      0.36      0.53     12598
           1       0.14      0.94      0.24      1402

    accuracy                           0.42     14000
   macro avg       0.56      0.65      0.38     14000
weighted avg       0.90      0.42      0.50     14000

[[4518 8080]
 [  87 1315]]
ROC-AUC: 0.6531512485621996


In [209]:
# Agregar a la tabla comparativa
resultados.append({
    "Modelo": "LR + Polynomial Features",
    "Accuracy": accuracy_score(y_val_no_sexo, y_pred_lr_poly),
    "Precision": precision_score(y_val_no_sexo, y_pred_lr_poly, zero_division=0),
    "Recall": recall_score(y_val_no_sexo, y_pred_lr_poly, zero_division=0),
    "F1": f1_score(y_val_no_sexo, y_pred_lr_poly, zero_division=0),
    "ROC_AUC": roc_lr_poly
})

resultados_df = (
    pd.DataFrame(resultados)
    .drop_duplicates(subset=["Modelo"], keep="last")
    .sort_values(by="ROC_AUC", ascending=False)
)

display(resultados_df)

,Modelo,Accuracy,Precision,Recall,F1,ROC_AUC
3,Logistic Regression - Escenario B,0.419571,0.140274,0.935093,0.243952,0.658912
12,Ablation - Modelo completo,0.419571,0.140274,0.935093,0.243952,0.658912
16,Ablation - Sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
20,Logistic Regression - Escenario B sin sexo,0.419500,0.140028,0.932953,0.243507,0.658860
25,LR + UnderSampling,0.420571,0.139944,0.930100,0.243284,0.657764
6,Random Forest - Escenario A,0.899683,0.000000,0.000000,0.000000,0.656499
23,LR Optimizada,0.418214,0.139989,0.935093,0.243522,0.656248
5,Decision Tree - Escenario B,0.412357,0.140070,0.947218,0.244050,0.655027
26,LR + Polynomial Features,0.416643,0.139968,0.937946,0.243586,0.653151
2,Logistic Regression - Escenario A,0.417280,0.140105,0.936017,0.243729,0.653076


##7. Modelo Final

In [210]:
from sklearn.metrics import accuracy_score
import numpy as np

thresholds = np.arange(
    0.01,
    0.99,
    0.01
)

best_acc = 0
best_thr_acc = 0

for thr in thresholds:

    pred = (
        y_prob_logreg_no_sexo >= thr
    ).astype(int)

    acc = accuracy_score(
        y_val_no_sexo,
        pred
    )

    if acc > best_acc:
        best_acc = acc
        best_thr_acc = thr

print(
    "Mejor threshold:",
    best_thr_acc
)

print(
    "Mejor Accuracy:",
    best_acc
)

Mejor threshold: 0.7100000000000001
Mejor Accuracy: 0.8998571428571429


In [211]:
y_pred_acc = (
    y_prob_logreg_no_sexo >= best_thr_acc
).astype(int)

print(
    classification_report(
        y_val_no_sexo,
        y_pred_acc
    )
)

print(
    confusion_matrix(
        y_val_no_sexo,
        y_pred_acc
    )
)

              precision    recall  f1-score   support

           0       0.90      1.00      0.95     12598
           1       0.50      0.00      0.00      1402

    accuracy                           0.90     14000
   macro avg       0.70      0.50      0.47     14000
weighted avg       0.86      0.90      0.85     14000

[[12597     1]
 [ 1401     1]]


In [212]:
feature_names = (
    preprocessor_no_sexo
    .get_feature_names_out()
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": logreg_no_sexo.coef_[0]
})

coef_df = coef_df.sort_values(
    "coef",
    ascending=False
)

coef_df.head(15)

,feature,coef
19,cat__nivel_educacion_madre_Primaria,0.920333
20,cat__nivel_educacion_madre_Secundaria,0.912646
22,cat__nivel_educacion_madre_Superior,0.881624
2,num__talla_cm,0.634037
1,num__peso_kg,0.404681
17,cat__region_Insular,0.316411
18,cat__region_Sierra,0.315553
16,cat__region_Costa,0.281916
11,num__imc,0.209228
4,num__talla_missing,0.036785


In [213]:
coef_df.tail(15)

,feature,coef
11,num__imc,0.209228
4,num__talla_missing,0.036785
14,num__talla_edad_ratio,0.016930
3,num__peso_missing,0.009033
9,num__talla_fuera_rango_oms,-0.014376
13,num__peso_edad_ratio,-0.016117
8,num__peso_fuera_rango_oms,-0.026911
10,num__peso_extremo,-0.110428
0,num__edad_meses,-0.276100
5,num__edad_clean,-0.276100


In [215]:
X_test.shape

(30000, 6)

In [217]:
X_test.head()

,edad_meses,sexo,region,nivel_educacion_madre,peso_kg,talla_cm
0,5,M,Insular,Secundaria,5.153938,56.287419
1,0,F,Amazonía,Secundaria,1.900358,51.444807
2,19,F,Amazonía,Primaria,10.297118,75.779276
3,20,M,Sierra,Secundaria,10.189172,NaN
4,6,M,Insular,Primaria,4.232316,56.278899


In [218]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'pd', 'np', 'px', 'go', 'train_test_split', 'StratifiedKFold', 'cross_val_score', 'RANDOM_STATE', '_i2', 'X_train', 'y_train', 'X_test', 'sample_submission', '_i3', '_i4', '_i5', 'df', '_5', '_i6', '_i7', 'data_types', '_i8', 'missing', '_i9', 'fig', '_i10', 'peso_missing_analysis', '_10', '_i11', 'talla_missing_analysis', '_11', '_i12', '_i13', 'edad_missing_analysis', '_13', '_i14', 'duplicados', '_i15', '_15', '_i16', 'duplicados_df', '_16', '_i17', 'categorical_features', 'col', '_i18', 'categorical_summary', '_i19', 'numerical_features', 'numeric_ranges', '_i20', 'numeric_summary', '_i21', 'rules_summary', '_i22', 'edad_menor_0', 'edad_mayor_24', 'peso_menor_2', 'peso_mayor_17', 'talla_menor_45', 'talla_mayor_97', 'inconsistencias_oms', '_i23', '_i24', 'quality_flags'

In [223]:
# ============================================================
# MODELO FINAL Y GENERACIÓN DE SUBMISSION PARA KAGGLE
# Modelo final: Logistic Regression - Escenario B sin sexo
# Criterio: alta sensibilidad para utilidad sanitaria
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. Función para reconstruir variables derivadas
# ------------------------------------------------------------

def preparar_variables_modelo(df_input):
    df_temp = df_input.copy()

    # Banderas de valores faltantes originales
    df_temp["peso_missing"] = df_temp["peso_kg"].isna().astype(int)
    df_temp["talla_missing"] = df_temp["talla_cm"].isna().astype(int)

    # Tratamiento de edad
    df_temp["edad_clean"] = df_temp["edad_meses"].where(
        (df_temp["edad_meses"] >= 0) &
        (df_temp["edad_meses"] <= 24),
        np.nan
    )

    df_temp["edad_missing"] = df_temp["edad_clean"].isna().astype(int)
    df_temp["edad_fuera_rango"] = (
        (df_temp["edad_meses"] < 0) |
        (df_temp["edad_meses"] > 24)
    ).astype(int)

    # Escenario B: corrección de pesos extremos asumiendo posible error decimal
    df_temp["peso_extremo"] = (df_temp["peso_kg"] > 30).astype(int)

    df_temp["peso_corregido"] = df_temp["peso_kg"].where(
        df_temp["peso_kg"] <= 30,
        df_temp["peso_kg"] / 10
    )

    df_temp["peso_kg"] = df_temp["peso_corregido"]

    # Banderas de plausibilidad
    df_temp["peso_fuera_rango_oms"] = (
        (df_temp["peso_kg"] < 2) |
        (df_temp["peso_kg"] > 17)
    ).astype(int)

    df_temp["talla_fuera_rango_oms"] = (
        (df_temp["talla_cm"] < 45) |
        (df_temp["talla_cm"] > 97)
    ).astype(int)

    # Variables derivadas
    df_temp["imc"] = (
        df_temp["peso_kg"] /
        ((df_temp["talla_cm"] / 100) ** 2)
    )

    df_temp["peso_talla_ratio"] = (
        df_temp["peso_kg"] /
        df_temp["talla_cm"]
    )

    df_temp["peso_edad_ratio"] = (
        df_temp["peso_kg"] /
        (df_temp["edad_clean"] + 1)
    )

    df_temp["talla_edad_ratio"] = (
        df_temp["talla_cm"] /
        (df_temp["edad_clean"] + 1)
    )

    # Eliminar columna auxiliar
    df_temp = df_temp.drop(columns=["peso_corregido"])

    return df_temp

In [224]:
# ------------------------------------------------------------
# 2. Preparar train completo y test Kaggle
# ------------------------------------------------------------

# df_B ya debe existir y contener el train procesado con alerta_desnutricion
X_full = df_B.drop(columns=["alerta_desnutricion", "sexo"])
y_full = df_B["alerta_desnutricion"]

# Preparar X_test con la misma lógica
X_test_B = preparar_variables_modelo(X_test)

# Eliminar sexo porque el modelo final no lo usa
X_test_final = X_test_B.drop(columns=["sexo"])

print("X_full:", X_full.shape)
print("X_test_final:", X_test_final.shape)
print("Columnas train:", X_full.columns.tolist())
print("Columnas test:", X_test_final.columns.tolist())

X_full: (70000, 17)
X_test_final: (30000, 17)
Columnas train: ['edad_meses', 'region', 'nivel_educacion_madre', 'peso_kg', 'talla_cm', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'peso_extremo', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']
Columnas test: ['edad_meses', 'region', 'nivel_educacion_madre', 'peso_kg', 'talla_cm', 'peso_missing', 'talla_missing', 'edad_clean', 'edad_missing', 'edad_fuera_rango', 'peso_extremo', 'peso_fuera_rango_oms', 'talla_fuera_rango_oms', 'imc', 'peso_talla_ratio', 'peso_edad_ratio', 'talla_edad_ratio']


In [225]:
# ------------------------------------------------------------
# 3. Verificar que train y test tengan las mismas columnas
# ------------------------------------------------------------

missing_in_test = set(X_full.columns) - set(X_test_final.columns)
extra_in_test = set(X_test_final.columns) - set(X_full.columns)

print("Columnas faltantes en test:", missing_in_test)
print("Columnas extra en test:", extra_in_test)

# Ordenar columnas del test igual que train
X_test_final = X_test_final[X_full.columns]

Columnas faltantes en test: set()
Columnas extra en test: set()


In [226]:
# ------------------------------------------------------------
# 4. Construir pipeline final
# ------------------------------------------------------------

cat_cols_final = X_full.select_dtypes(include="object").columns.tolist()

num_cols_final = [
    col for col in X_full.columns
    if col not in cat_cols_final
]

numeric_transformer_final = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]
)

categorical_transformer_final = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_final = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_final, num_cols_final),
        ("cat", categorical_transformer_final, cat_cols_final)
    ]
)

modelo_final = Pipeline(
    steps=[
        ("preprocessor", preprocessor_final),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

In [227]:
# ------------------------------------------------------------
# 5. Entrenar modelo final con todo train
# ------------------------------------------------------------

modelo_final.fit(X_full, y_full)

print("Modelo final entrenado.")

Modelo final entrenado.


In [228]:
# ------------------------------------------------------------
# 6. Generar probabilidades y predicciones
# ------------------------------------------------------------

y_test_prob = modelo_final.predict_proba(X_test_final)[:, 1]

# Umbral elegido por utilidad sanitaria
THRESHOLD_FINAL = 0.50

y_test_pred = (y_test_prob >= THRESHOLD_FINAL).astype(int)

print(pd.Series(y_test_prob).describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

print(pd.Series(y_test_pred).value_counts(normalize=True))
print(pd.Series(y_test_pred).value_counts())

count    30000.000000
mean         0.446100
std          0.214042
min          0.000384
50%          0.571439
75%          0.602892
90%          0.624391
95%          0.636522
99%          0.659373
max          0.761896
dtype: float64
1    0.6486
0    0.3514
Name: proportion, dtype: float64
1    19458
0    10542
Name: count, dtype: int64


In [229]:
# ------------------------------------------------------------
# 7. Revisar sample_submission
# ------------------------------------------------------------

print(sample_submission.head())
print(sample_submission.shape)
print(sample_submission.columns.tolist())

   ID  alerta_desnutricion
0   1                    0
1   2                    0
2   3                    0
3   4                    0
4   5                    0
(30000, 2)
['ID', 'alerta_desnutricion']


In [230]:
# ------------------------------------------------------------
# 8. Crear archivo submission
# ------------------------------------------------------------

submission = sample_submission.copy()

# Detectar la columna objetivo automáticamente
target_col = [
    col for col in submission.columns
    if col != submission.columns[0]
][0]

submission[target_col] = y_test_pred

print(submission.head())
print(submission[target_col].value_counts())

submission.to_csv(
    "submission_final.csv",
    index=False
)

print("Archivo generado: submission_final.csv")

   ID  alerta_desnutricion
0   1                    1
1   2                    0
2   3                    0
3   4                    1
4   5                    1
alerta_desnutricion
1    19458
0    10542
Name: count, dtype: int64
Archivo generado: submission_final.csv


In [231]:
pd.Series(y_test_prob).describe(
    percentiles=[0.5,0.75,0.9,0.95,0.99]
)

,0
count,30000.000000
mean,0.446100
std,0.214042
min,0.000384
50%,0.571439
75%,0.602892
90%,0.624391
95%,0.636522
99%,0.659373
max,0.761896


In [232]:
y_test_pred_60 = (y_test_prob >= 0.60).astype(int)

submission_60 = sample_submission.copy()
submission_60["alerta_desnutricion"] = y_test_pred_60

submission_60.to_csv(
    "submission_threshold_060.csv",
    index=False
)

In [233]:
y_test_pred_70 = (y_test_prob >= 0.70).astype(int)

submission_70 = sample_submission.copy()
submission_70["alerta_desnutricion"] = y_test_pred_70

submission_70.to_csv(
    "submission_threshold_070.csv",
    index=False
)

In [234]:
print("Threshold 0.50")
print(pd.Series(y_test_pred).value_counts())

print("Threshold 0.60")
print(pd.Series(y_test_pred_60).value_counts())

print("Threshold 0.70")
print(pd.Series(y_test_pred_70).value_counts())

Threshold 0.50
1    19458
0    10542
Name: count, dtype: int64
Threshold 0.60
0    21724
1     8276
Name: count, dtype: int64
Threshold 0.70
0    29978
1       22
Name: count, dtype: int64


In [235]:
for thr in [0.55, 0.58, 0.62, 0.65]:

    pred = (y_test_prob >= thr).astype(int)

    sub = sample_submission.copy()
    sub["alerta_desnutricion"] = pred

    sub.to_csv(
        f"submission_thr_{str(thr).replace('.','')}.csv",
        index=False
    )

    print(
        thr,
        pred.sum(),
        len(pred) - pred.sum()
    )

0.55 17802 12198
0.58 13200 16800
0.62 3757 26243
0.65 623 29377
